In [2]:
import re
import os
import numpy as np
import pandas as pd
from datetime import datetime

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)


In [3]:
DATA_PATH = r"C:\Users\S. Umredkar\Desktop\Data Science IPCS Global\Resume screening Tool\Dataset\Resume\Resume_cleaned.csv"

df = pd.read_csv(DATA_PATH)

print(df.shape)
print(df.columns.tolist())

(2481, 5)
['ID', 'Resume_str', 'Resume_html', 'Category', 'Cleaned_Resume']


In [4]:
print("Rows:", len(df))
print("Categories:", df["Category"].nunique())
print("Missing Cleaned_Resume:", df["Cleaned_Resume"].isna().sum())

Rows: 2481
Categories: 24
Missing Cleaned_Resume: 0


In [5]:
job_profile = {
    "job_title": "Data Scientist",
    "domain": "Data Science",
    "required_skills": [
        "python",
        "sql",
        "pandas",
        "numpy",
        "data analysis",
        "statistics",
        "data visualization",
        "machine learning",
        "predictive modeling",
        "scikit-learn"
    ],
    "preferred_skills": [],
    "job_relevance_terms": [
        "data science",
        "data scientist",
        "data analyst",
        "data analysis",
        "statistical modeling",
        "machine learning",
        "predictive modeling"
    ]
}

required_experience = 5

required_education = [
    "data science",
    "statistics",
    "mathematics",
    "computer science",
    "information technology"
]

print(job_profile)

{'job_title': 'Data Scientist', 'domain': 'Data Science', 'required_skills': ['python', 'sql', 'pandas', 'numpy', 'data analysis', 'statistics', 'data visualization', 'machine learning', 'predictive modeling', 'scikit-learn'], 'preferred_skills': [], 'job_relevance_terms': ['data science', 'data scientist', 'data analyst', 'data analysis', 'statistical modeling', 'machine learning', 'predictive modeling']}


In [62]:
def extract_experience(resume_text):
    text = str(resume_text).lower()

    # --------------------------------------------------
    # 1. Explicit professional/work experience statements
    # --------------------------------------------------
    patterns = [
        r'(\d+(?:\.\d+)?)\+?\s*(?:years?|yrs?)\s+(?:of\s+)?(?:professional\s+)?(?:work\s+)?experience',
        r'(\d+(?:\.\d+)?)\+?\s*(?:years?|yrs?)\s+(?:of\s+)?(?:relevant\s+)?experience',
        r'(\d+(?:\.\d+)?)\+?\s*(?:years?|yrs?)\s+(?:in|of)\s+(?:professional\s+)?(?:work|employment)',
        r'(?:experience|worked|working)\s+(?:for|of|with)?\s*(\d+(?:\.\d+)?)\+?\s*(?:years?|yrs?)'
    ]

    values = []

    for pattern in patterns:
        matches = re.findall(pattern, text)

        for match in matches:
            try:
                values.append(float(match))
            except:
                pass

    if values:
        return round(max(values), 2)

    # --------------------------------------------------
    # 2. Years + months experience
    # --------------------------------------------------
    year_month_pattern = (
        r'(\d+(?:\.\d+)?)\s*years?'
        r'(?:\s*(?:and|,)?\s*(\d+)\s*months?)?'
    )

    matches = re.findall(year_month_pattern, text)

    for years, months in matches:

        # Ignore suspiciously low-context matches
        # unless "experience/work" occurs nearby
        values.append(
            float(years) +
            (float(months) / 12 if months else 0)
        )

    if values:
        return round(max(values), 2)

    # --------------------------------------------------
    # 3. Month-Year → Current/Present
    # --------------------------------------------------
    month_pattern = (
        r'(jan(?:uary)?|feb(?:ruary)?|mar(?:ch)?|apr(?:il)?|'
        r'may|jun(?:e)?|jul(?:y)?|aug(?:ust)?|sep(?:tember)?|'
        r'sept(?:ember)?|oct(?:ober)?|nov(?:ember)?|'
        r'dec(?:ember)?)'
        r'\s+(\d{4})\s*(?:-|–|—|to)\s*(current|present)'
    )

    matches = re.findall(month_pattern, text)

    if matches:

        month_map = {
            "jan": 1, "january": 1,
            "feb": 2, "february": 2,
            "mar": 3, "march": 3,
            "apr": 4, "april": 4,
            "may": 5,
            "jun": 6, "june": 6,
            "jul": 7, "july": 7,
            "aug": 8, "august": 8,
            "sep": 9, "september": 9,
            "sept": 9,
            "oct": 10, "october": 10,
            "nov": 11, "november": 11,
            "dec": 12, "december": 12
        }

        current_date = datetime.now()
        values = []

        for month, year, _ in matches:

            start = datetime(
                int(year),
                month_map[month],
                1
            )

            months = (
                (current_date.year - start.year) * 12
                + current_date.month - start.month
            )

            values.append(months / 12)

        if values:
            return round(max(values), 2)

    # --------------------------------------------------
    # 4. Year → Current/Present
    # --------------------------------------------------
    year_current_pattern = (
        r'(\d{4})\s*(?:-|–|—|to)\s*(?:current|present)'
    )

    matches = re.findall(
        year_current_pattern,
        text
    )

    if matches:

        current_year = datetime.now().year

        return round(
            max(
                current_year - int(year)
                for year in matches
            ),
            2
        )

    # --------------------------------------------------
    # 5. Nothing found
    # --------------------------------------------------
    return 0.0

In [63]:
test_experience = {
    "5 years of experience": 5.0,
    "3 years of professional experience": 3.0,
    "10+ years of relevant experience": 10.0,
    "worked for 7 years": 7.0,
    "No experience mentioned": 0.0,
    "B.Tech completed in 2019": 0.0
}

for text, expected in test_experience.items():
    result = extract_experience(text)
    print(f"{text:45} → {result} | Expected: {expected}")

5 years of experience                         → 5.0 | Expected: 5.0
3 years of professional experience            → 3.0 | Expected: 3.0
10+ years of relevant experience              → 10.0 | Expected: 10.0
worked for 7 years                            → 7.0 | Expected: 7.0
No experience mentioned                       → 0.0 | Expected: 0.0
B.Tech completed in 2019                      → 0.0 | Expected: 0.0


In [136]:
def extract_education(resume_text):
    text = str(resume_text).lower()

    degrees = []
    fields = []

    # ==================================================
    # 1. Extract EDUCATION section
    # ==================================================

    education_pattern = (
        r'\beducation\b'
        r'(.*?)(?=\bskills\b|\bexperience\b|\bprojects\b|$)'
    )

    education_match = re.search(
        education_pattern,
        text,
        flags=re.IGNORECASE | re.DOTALL
    )

    if education_match:
        education_text = education_match.group(1)
    else:
        education_text = ""

    # ==================================================
    # 2. Degree patterns
    # ==================================================

    degree_patterns = {
        "b.e": [
            r'\bb\.?e\.?\b',
            r'\bbachelors?\s+of\s+engineering\b'
        ],

        "b.tech": [
            r'\bb\.?tech\.?\b',
            r'\bbtech\b',
            r'\bbachelors?\s+of\s+technology\b'
        ],

        "b.sc": [
            r'\bb\.?sc\.?\b',
            r'\bbsc\b',
            r'\bbachelors?\s+of\s+science\b'
        ],

        "m.e": [
            r'\bm\.?e\.?\b',
            r'\bmasters?\s+of\s+engineering\b'
        ],

        "m.tech": [
            r'\bm\.?tech\.?\b',
            r'\bmtech\b',
            r'\bmasters?\s+of\s+technology\b'
        ],

        "m.sc": [
            r'\bm\.?sc\.?\b',
            r'\bmsc\b',
            r'\bmasters?\s+of\s+science\b'
        ],

        "mba": [
            r'\bmba\b',
            r'\bmasters?\s+of\s+business\s+administration\b'
        ],

        "ph.d": [
            r'\bph\.?d\.?\b',
            r'\bdoctorate\b',
            r'\bdoctor\s+of\s+philosophy\b'
        ]
    }

    # ==================================================
    # 3. Find degrees
    # ==================================================

    degree_positions = []

    for degree, patterns in degree_patterns.items():

        for pattern in patterns:

            matches = list(
                re.finditer(
                    pattern,
                    education_text,
                    flags=re.IGNORECASE
                )
            )

            if matches:

                degrees.append(degree)

                degree_positions.extend(matches)

                break

    # ==================================================
    # 4. Education field patterns
    # ==================================================

    field_patterns = [
        "data science",
        "computer science",
        "information technology",
        "information systems",
        "statistics",
        "mathematics",
        "machine learning",
        "artificial intelligence",
        "software engineering",
        "mechanical engineering",
        "electrical engineering",
        "civil engineering",
        "industrial and systems engineering",
        "business administration",
        "finance",
        "marketing"
    ]

    # ==================================================
    # 5. Search fields near degree
    # ==================================================

    for degree_match in degree_positions:

        start = max(
            0,
            degree_match.start() - 100
        )

        end = min(
            len(education_text),
            degree_match.end() + 300
        )

        context = education_text[start:end]

        for field in field_patterns:

            if re.search(
                rf'\b{re.escape(field)}\b',
                context
            ):
                fields.append(field)

    return {
        "degrees": sorted(set(degrees)),
        "fields": sorted(set(fields))
    }

In [138]:
test_ids = [12011623, 18448085, 10265057]

for candidate_id in test_ids:

    resume_text = df.loc[
        df["ID"] == candidate_id,
        "Resume_str"
    ].iloc[0]

    print("\nID:", candidate_id)
    print(extract_education(resume_text))


ID: 12011623
{'degrees': [], 'fields': []}

ID: 18448085
{'degrees': ['b.e', 'm.sc'], 'fields': ['industrial and systems engineering', 'mechanical engineering']}

ID: 10265057
{'degrees': [], 'fields': []}


In [135]:
education_pattern = (
    r'\beducation\b'
    r'(.*?)(?=\bskills\b|\bexperience\b|\bprojects\b|$)'
)

match = re.search(
    education_pattern,
    text,
    flags=re.IGNORECASE | re.DOTALL
)

if match:
    education_text = match.group(1)

    print("===== EDUCATION TEXT =====")
    print(education_text[:2000])
else:
    print("NO MATCH")

===== EDUCATION TEXT =====
     aug 2016     masters of science  :   industrial and systems engineering    binghamton university, state university of new york          industrial and systems engineering 3.51/4.00       may 2014     bachelors of engineering  :   mechanical engineering    osmania university          mechanical engineering 3.33/4.00          


In [206]:
EDUCATION_DIRECT = {
    "data science",
    "statistics",
    "mathematics",
    "computer science",
    "information technology"
}

EDUCATION_RELATED = {
    "machine learning",
    "computer engineering",
    "electrical and computer engineering",
    "industrial and systems engineering"
}


def calculate_education_match(candidate_education, required_education):

    if not isinstance(candidate_education, dict):
        return 0.0

    fields = {
        field.lower().strip()
        for field in candidate_education.get("fields", [])
    }

    if not fields:
        return 0.0

    # Direct match
    if fields.intersection(EDUCATION_DIRECT):
        return 100.0

    # Related field
    if fields.intersection(EDUCATION_RELATED):
        return 60.0

    return 0.0

In [207]:
required_education = [
    'data science',
    'statistics',
    'mathematics',
    'computer science',
    'information technology'
]

test_education = {
    'degrees': ['b.sc', 'm.sc'],
    'fields': ['computer engineering']
}

print(calculate_education_match(test_education, required_education))

60.0


In [208]:
EDUCATION_ALIASES = {
    "computer engineering": {
        "computer engineering",
        "computer science",
        "information technology"
    },

    "electrical and computer engineering": {
        "computer engineering",
        "computer science",
        "information technology"
    },

    "industrial and systems engineering": {
        "statistics",
        "mathematics",
        "data science"
    },

    "machine learning": {
        "data science",
        "computer science"
    }
}

In [209]:
def normalize_education_fields(fields):
    normalized = set()

    for field in fields:
        field = field.lower().strip()

        normalized.add(field)

        if field in EDUCATION_ALIASES:
            normalized.update(EDUCATION_ALIASES[field])

    return normalized

In [212]:
test_ids = [21156767, 12011623, 18448085, 10265057]

for candidate_id in test_ids:

    row = df[df["ID"] == candidate_id].iloc[0]

    education = extract_education(row["Resume_str"])

    score = calculate_education_match(
        education,
        required_education
    )

    print("=" * 60)
    print("ID:", candidate_id)
    print("Education:", education)
    print("Education Match:", score)

ID: 21156767
Education: {'degrees': ['b.e', 'm.sc'], 'fields': ['data science', 'computer science', 'machine learning']}
Education Match: 100.0
ID: 12011623
Education: {'degrees': [], 'fields': ['machine learning']}
Education Match: 60.0
ID: 18448085
Education: {'degrees': ['b.e', 'm.sc'], 'fields': ['mechanical engineering', 'industrial and systems engineering']}
Education Match: 60.0
ID: 10265057
Education: {'degrees': ['b.sc', 'm.sc'], 'fields': ['computer engineering']}
Education Match: 60.0


In [211]:
def detailed_skill_match(resume_text, required_skills, preferred_skills=None):
    if preferred_skills is None:
        preferred_skills = []

    text = str(resume_text).lower()

    matched_required = []
    missing_required = []

    for skill in required_skills:
        if skill.lower() in text:
            matched_required.append(skill)
        else:
            missing_required.append(skill)

    matched_preferred = [
        skill for skill in preferred_skills
        if skill.lower() in text
    ]

    required_score = (
        len(matched_required) / len(required_skills) * 100
        if required_skills else 100
    )

    preferred_score = (
        len(matched_preferred) / len(preferred_skills) * 100
        if preferred_skills else 0
    )

    return {
        "matched_required": matched_required,
        "missing_required": missing_required,
        "matched_preferred": matched_preferred,
        "required_score": round(required_score, 2),
        "preferred_score": round(preferred_score, 2)
    }


def calculate_final_skill_score(skill_details):

    # Importance weights for this Data Scientist JD
    skill_weights = {
        "python": 15,
        "sql": 8,
        "pandas": 8,
        "numpy": 5,
        "data analysis": 8,
        "statistics": 12,
        "data visualization": 5,
        "machine learning": 15,
        "predictive modeling": 14,
        "scikit-learn": 10
    }

    matched_skills = [
        skill.lower().strip()
        for skill in skill_details["matched_required"]
    ]

    total_weight = sum(skill_weights.values())

    matched_weight = sum(
        skill_weights.get(skill, 0)
        for skill in matched_skills
    )

    score = (
        matched_weight / total_weight * 100
        if total_weight > 0 else 0
    )

    return round(score, 2)

In [85]:
skill_test = detailed_skill_match(
    df.iloc[0]["Cleaned_Resume"],
    job_profile["required_skills"],
    job_profile["preferred_skills"]
)

print("Matched:", skill_test["matched_required"])
print("Missing:", skill_test["missing_required"])
print("Required Score:", skill_test["required_score"])

Matched: ['data analysis', 'statistics']
Missing: ['python', 'sql', 'pandas', 'numpy', 'data visualization', 'machine learning', 'predictive modeling', 'scikit-learn']
Required Score: 20.0


In [86]:
def extract_job_relevance(resume_text, job_profile):

    text = str(resume_text).lower()

    relevance_terms = [
        term.lower().strip()
        for term in job_profile.get("job_relevance_terms", [])
        if term
    ]

    if not relevance_terms:
        return 0.0, []

    matched_terms = [
        term
        for term in relevance_terms
        if term in text
    ]

    score = (
        len(matched_terms) / len(relevance_terms)
    ) * 100

    return round(score, 2), matched_terms

In [87]:
test_resume = df.iloc[0]["Resume_str"]

relevance_score, matched_terms = extract_job_relevance(
    test_resume,
    job_profile
)

print("Relevance Score:", relevance_score)
print("Matched Terms:", matched_terms)

Relevance Score: 14.29
Matched Terms: ['data analysis']


In [88]:
test_id = ranking_df.iloc[0]["ID"]

test_resume = df.loc[
    df["ID"] == test_id,
    "Resume_str"
].iloc[0]

relevance_score, matched_terms = extract_job_relevance(
    test_resume,
    job_profile
)

print("Candidate ID:", test_id)
print("Relevance Score:", relevance_score)
print("Matched Terms:", matched_terms)

Candidate ID: 18448085
Relevance Score: 57.14
Matched Terms: ['data science', 'data analyst', 'data analysis', 'statistical modeling']


In [89]:
def screen_candidate(
    resume_text,
    required_skills,
    preferred_skills,
    required_experience,
    required_education,
    job_profile
):
    experience = extract_experience(resume_text)

    experience_match = (
        min((experience / required_experience) * 100, 100)
        if required_experience > 0 else 100
    )

    education = extract_education(resume_text)

    education_match = calculate_education_match(
        education,
        required_education
    )

    skill_details = detailed_skill_match(
        resume_text,
        required_skills,
        preferred_skills
    )

    skills_match = calculate_final_skill_score(skill_details)

    relevance_score, relevant_terms = extract_job_relevance(
        resume_text,
        job_profile
    )

    overall_match = round(
        experience_match * 0.25 +
        education_match * 0.20 +
        skills_match * 0.40 +
        relevance_score * 0.15,
        2
    )

    return {
        "experience": experience,
        "experience_match": round(experience_match, 2),
        "education": education,
        "education_match": round(education_match, 2),
        "skill_details": skill_details,
        "skills_match": round(skills_match, 2),
        "relevant_terms": relevant_terms,
        "job_relevance": relevance_score,
        "overall_match": overall_match
    }

In [90]:
result = screen_candidate(
    df.iloc[0]["Cleaned_Resume"],
    job_profile["required_skills"],
    job_profile["preferred_skills"],
    required_experience,
    required_education,
    job_profile
)

print("Experience:", result["experience"])
print("Experience Match:", result["experience_match"])
print("Education Match:", result["education_match"])
print("Skills Match:", result["skills_match"])
print("Job Relevance:", result["job_relevance"])
print("Overall Match:", result["overall_match"])

Experience: 15.0
Experience Match: 100
Education Match: 100.0
Skills Match: 20.0
Job Relevance: 14.29
Overall Match: 55.14


In [147]:
results = []

for _, row in df.iterrows():

    result = screen_candidate(
        resume_text=row["Resume_str"],
        required_skills=job_profile["required_skills"],
        preferred_skills=job_profile["preferred_skills"],
        required_experience=required_experience,
        required_education=required_education,
        job_profile=job_profile
    )

    results.append({
        "ID": row["ID"],
        "Category": row["Category"],
        "Experience": result["experience"],
        "Experience Match": result["experience_match"],
        "Education Match": result["education_match"],
        "Skills Match": result["skills_match"],
        "Job Relevance": result["job_relevance"],
        "Overall Match": result["overall_match"],
        "Matched Skills": ", ".join(
            result["skill_details"]["matched_required"]
        ),
        "Missing Skills": ", ".join(
            result["skill_details"]["missing_required"]
        ),
        "Relevant Terms": ", ".join(
            result["relevant_terms"]
        ),
        "Explanation": generate_explanation(
            result,
            required_experience
        )
    })

ranking_df = pd.DataFrame(results)

ranking_df = ranking_df.sort_values(
    "Overall Match",
    ascending=False
).reset_index(drop=True)

ranking_df.insert(
    0,
    "Rank",
    range(1, len(ranking_df) + 1)
)

ranking_df["Recommendation"] = (
    ranking_df["Overall Match"]
    .apply(get_recommendation)
)

In [148]:
print("===== FINAL SCORE STATISTICS =====")
print(ranking_df["Overall Match"].describe())

print("\n===== FINAL RECOMMENDATION DISTRIBUTION =====")
print(ranking_df["Recommendation"].value_counts())

===== FINAL SCORE STATISTICS =====
count    2481.000000
mean       21.600544
std        12.024207
min         0.000000
25%        16.000000
50%        25.000000
75%        29.000000
max        71.830000
Name: Overall Match, dtype: float64

===== FINAL RECOMMENDATION DISTRIBUTION =====
Recommendation
Weak Match        2458
Moderate Match      22
Good Match           1
Name: count, dtype: int64


In [149]:
display(
    ranking_df[
        [
            "Rank",
            "ID",
            "Category",
            "Experience",
            "Experience Match",
            "Education Match",
            "Skills Match",
            "Job Relevance",
            "Overall Match",
            "Recommendation",
            "Matched Skills",
            "Missing Skills",
            "Relevant Terms"
        ]
    ].head(15)
)

,Rank,ID,Category,Experience,Experience Match,Education Match,Skills Match,Job Relevance,Overall Match,Recommendation,Matched Skills,Missing Skills,Relevant Terms
0,1,21156767,CONSULTANT,6.00,100.0,100.0,51.0,42.86,71.83,Good Match,"python, sql, data analysis, data visualization, machine learning","pandas, numpy, statistics, predictive modeling, scikit-learn","data science, data analysis, machine learning"
1,2,18448085,AUTOMOBILE,10.00,100.0,20.0,61.0,57.14,61.97,Moderate Match,"python, sql, pandas, numpy, data analysis, statistics, data visualization","machine learning, predictive modeling, scikit-learn","data science, data analyst, data analysis, statistical modeling"
2,3,29839396,BANKING,14.75,100.0,100.0,35.0,14.29,61.14,Moderate Match,"python, data analysis, statistics","sql, pandas, numpy, data visualization, machine learning, predictive modeling, scikit-learn",data analysis
3,4,11813872,AGRICULTURE,20.00,100.0,100.0,31.0,14.29,59.54,Moderate Match,"python, sql, pandas","numpy, data analysis, statistics, data visualization, machine learning, predictive modeling, sci...",data science
4,5,12011623,ENGINEERING,3.00,60.0,0.0,85.0,57.14,57.57,Moderate Match,"python, sql, pandas, data analysis, statistics, data visualization, machine learning, predictive...","numpy, scikit-learn","data analyst, data analysis, machine learning, predictive modeling"
5,6,22946204,AUTOMOBILE,10.17,100.0,20.0,58.0,28.57,56.49,Moderate Match,"python, sql, data analysis, statistics, machine learning","pandas, numpy, data visualization, predictive modeling, scikit-learn","data analysis, machine learning"
6,7,12635195,INFORMATION-TECHNOLOGY,15.00,100.0,100.0,23.0,14.29,56.34,Moderate Match,"python, data analysis","sql, pandas, numpy, statistics, data visualization, machine learning, predictive modeling, sciki...",data analysis
7,8,50328713,ENGINEERING,13.00,100.0,20.0,56.0,28.57,55.69,Moderate Match,"python, sql, pandas, machine learning, scikit-learn","numpy, data analysis, statistics, data visualization, predictive modeling","statistical modeling, machine learning"
8,9,47410104,FINANCE,16.75,100.0,100.0,20.0,14.29,55.14,Moderate Match,"data analysis, statistics","python, sql, pandas, numpy, data visualization, machine learning, predictive modeling, scikit-learn",data analysis
9,10,12526702,CONSULTANT,15.00,100.0,100.0,20.0,14.29,55.14,Moderate Match,"data analysis, statistics","python, sql, pandas, numpy, data visualization, machine learning, predictive modeling, scikit-learn",data analysis


In [258]:
def get_recommendation(score):
    if score >= 70:
        return "Good Match"
    elif score >= 50:
        return "Moderate Match"
    else:
        return "Weak Match"

ranking_df["Recommendation"] = (
    ranking_df["Overall Match"]
    .apply(get_recommendation)
)

In [259]:
ranking_df = (
    ranking_df
    .sort_values("Overall Match", ascending=False)
    .reset_index(drop=True)
)

ranking_df["Rank"] = ranking_df.index + 1

In [261]:
print("===== FINAL SCORE STATISTICS =====")
print(ranking_df["Overall Match"].describe())

print("\n===== RECOMMENDATION DISTRIBUTION =====")
print(ranking_df["Recommendation"].value_counts())

print("\n===== TOP 15 CANDIDATES =====")

display(
    ranking_df.head(15)[
        [
            "Rank",
            "ID",
            "Category",
            "Experience Match",
            "Education Match",
            "Skills Match",
            "Job Relevance",
            "Overall Match",
            "Recommendation",
            "Matched Skills",
            "Missing Skills"
        ]
    ]
)

===== FINAL SCORE STATISTICS =====
count    2481.000000
mean       17.648408
std        10.913242
min         0.000000
25%        14.000000
50%        20.000000
75%        20.000000
max        57.430000
Name: Overall Match, dtype: float64

===== RECOMMENDATION DISTRIBUTION =====
Recommendation
Weak Match        2477
Moderate Match       4
Name: count, dtype: int64

===== TOP 15 CANDIDATES =====


,Rank,ID,Category,Experience Match,Education Match,Skills Match,Job Relevance,Overall Match,Recommendation,Matched Skills,Missing Skills
0,1,12011623,ENGINEERING,60.0,0.0,85.0,57.14,57.43,Moderate Match,"python, sql, pandas, data analysis, statistics, data visualization, machine learning, predictive...","numpy, scikit-learn"
1,2,18448085,AUTOMOBILE,100.0,0.0,61.0,57.14,55.83,Moderate Match,"python, sql, pandas, numpy, data analysis, statistics, data visualization","machine learning, predictive modeling, scikit-learn"
2,3,20674668,INFORMATION-TECHNOLOGY,100.0,100.0,35.0,0.00,54.00,Moderate Match,"python, sql, statistics","pandas, numpy, data analysis, data visualization, machine learning, predictive modeling, scikit-..."
3,4,11332602,FITNESS,100.0,100.0,20.0,14.29,50.86,Moderate Match,"data analysis, statistics","python, sql, pandas, numpy, data visualization, machine learning, predictive modeling, scikit-learn"
4,5,21156767,CONSULTANT,100.0,0.0,51.0,42.86,48.97,Weak Match,"python, sql, data analysis, data visualization, machine learning","pandas, numpy, statistics, predictive modeling, scikit-learn"
5,6,22946204,AUTOMOBILE,100.0,0.0,58.0,28.57,48.91,Weak Match,"python, sql, data analysis, statistics, machine learning","pandas, numpy, data visualization, predictive modeling, scikit-learn"
6,7,50328713,ENGINEERING,100.0,0.0,56.0,28.57,48.11,Weak Match,"python, sql, pandas, machine learning, scikit-learn","numpy, data analysis, statistics, data visualization, predictive modeling"
7,8,86549455,FINANCE,100.0,100.0,8.0,14.29,46.06,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin..."
8,9,27847081,HR,100.0,100.0,8.0,14.29,46.06,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin..."
9,10,15941675,ENGINEERING,100.0,100.0,8.0,14.29,46.06,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin..."


In [262]:
ranking_df["Eligibility"] = "Eligible"

ranking_df.loc[
    ranking_df["Experience Match"] < 100,
    "Eligibility"
] = "Experience Gap"

ranking_df.loc[
    (ranking_df["Experience Match"] < 100) &
    (ranking_df["Education Match"] < 100),
    "Eligibility"
] = "Experience + Education Gap"

ranking_df.loc[
    (ranking_df["Experience Match"] >= 100) &
    (ranking_df["Education Match"] < 100),
    "Eligibility"
] = "Education Gap"

In [263]:
print("===== ELIGIBILITY DISTRIBUTION =====")
print(ranking_df["Eligibility"].value_counts())

===== ELIGIBILITY DISTRIBUTION =====
Eligibility
Education Gap                 1590
Experience + Education Gap     677
Eligible                       158
Experience Gap                  56
Name: count, dtype: int64


In [264]:
eligible_df = (
    ranking_df[
        ranking_df["Eligibility"] == "Eligible"
    ]
    .sort_values("Overall Match", ascending=False)
    .reset_index(drop=True)
)

eligible_df["Eligible Rank"] = eligible_df.index + 1

print("Eligible candidates:", len(eligible_df))

display(
    eligible_df.head(15)[
        [
            "Eligible Rank",
            "ID",
            "Category",
            "Experience Match",
            "Education Match",
            "Skills Match",
            "Job Relevance",
            "Overall Match",
            "Recommendation",
            "Matched Skills",
            "Missing Skills"
        ]
    ]
)

Eligible candidates: 158


,Eligible Rank,ID,Category,Experience Match,Education Match,Skills Match,Job Relevance,Overall Match,Recommendation,Matched Skills,Missing Skills
0,1,20674668,INFORMATION-TECHNOLOGY,100.0,100.0,35.0,0.00,54.00,Moderate Match,"python, sql, statistics","pandas, numpy, data analysis, data visualization, machine learning, predictive modeling, scikit-..."
1,2,11332602,FITNESS,100.0,100.0,20.0,14.29,50.86,Moderate Match,"data analysis, statistics","python, sql, pandas, numpy, data visualization, machine learning, predictive modeling, scikit-learn"
2,3,86549455,FINANCE,100.0,100.0,8.0,14.29,46.06,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin..."
3,4,27847081,HR,100.0,100.0,8.0,14.29,46.06,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin..."
4,5,15941675,ENGINEERING,100.0,100.0,8.0,14.29,46.06,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin..."
5,6,16121387,HEALTHCARE,100.0,100.0,8.0,14.29,46.06,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin..."
6,7,91539554,DIGITAL-MEDIA,100.0,100.0,8.0,14.29,46.06,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin..."
7,8,17412079,HR,100.0,100.0,8.0,14.29,46.06,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin..."
8,9,41910253,HEALTHCARE,100.0,100.0,8.0,14.29,46.06,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin..."
9,10,11432686,CHEF,100.0,100.0,8.0,14.29,46.06,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin..."


In [ ]:
def explain_candidate(result):
    experience = result["experience"]
    experience_match = result["experience_match"]
    education_match = result["education_match"]
    skills = result["skill_details"]
    relevance = result["job_relevance"]
    terms = result["relevant_terms"]

    explanation = []

    if experience_match >= 100:
        explanation.append(
            f"Meets the required experience with {experience} years of experience."
        )
    else:
        explanation.append(
            f"Has {experience} years of experience against the required "
            f"{required_experience} years."
        )

    if education_match == 100:
        explanation.append("Education is highly aligned with the job requirements.")
    elif education_match >= 70:
        explanation.append("Education is partially aligned with the job requirements.")
    elif education_match > 0:
        explanation.append("A degree was detected, but the field is not closely aligned.")
    else:
        explanation.append("No relevant education match was detected.")

    explanation.append(
        f"Matches {len(skills['matched_required'])} required skills."
    )

    if skills["missing_required"]:
        explanation.append(
            "Missing key skills: " +
            ", ".join(skills["missing_required"]) + "."
        )

    if terms:
        explanation.append(
            "Relevant terms found: " + ", ".join(terms) + "."
        )

    return " ".join(explanation)

In [ ]:
top_id = ranking_df.iloc[0]["ID"]

top_resume = df.loc[
    df["ID"] == top_id,
    "Cleaned_Resume"
].iloc[0]

top_result = screen_candidate(
    top_resume,
    job_profile["required_skills"],
    job_profile["preferred_skills"],
    required_experience,
    required_education,
    job_profile
)

print("===== TOP CANDIDATE =====")
print("ID:", top_id)
print("Overall Match:", top_result["overall_match"])
print()
print(explain_candidate(top_result))

In [ ]:
explanations = []

for _, row in ranking_df.iterrows():
    resume = df.loc[
        df["ID"] == row["ID"],
        "Cleaned_Resume"
    ].iloc[0]

    result = screen_candidate(
        resume,
        job_profile["required_skills"],
        job_profile["preferred_skills"],
        required_experience,
        required_education,
        job_profile
    )

    explanations.append(explain_candidate(result))

ranking_df["Explanation"] = explanations

print(ranking_df[
    ["Rank", "ID", "Category", "Overall Match",
     "Recommendation", "Explanation"]
].head(5).to_string(index=False))

In [ ]:
ranking_df.to_csv("final_candidate_ranking.csv", index=False)

print("Saved:", "final_candidate_ranking.csv")
print("Rows:", len(ranking_df))

In [ ]:
print("===== SCORE STATISTICS =====")
print(ranking_df["Overall Match"].describe())

print("\n===== RECOMMENDATION DISTRIBUTION =====")
print(ranking_df["Recommendation"].value_counts())

In [ ]:
ranking_df[
    [
        "Rank",
        "ID",
        "Category",
        "Experience Match",
        "Education Match",
        "Skills Match",
        "Job Relevance",
        "Overall Match",
        "Recommendation"
    ]
].head(20)

In [ ]:
ranking_df["Category"].value_counts().head(10)

In [223]:
ranking_df.sort_values(
    "Job Relevance",
    ascending=False
)[
    [
        "Rank",
        "ID",
        "Category",
        "Experience Match",
        "Education Match",
        "Skills Match",
        "Job Relevance",
        "Overall Match",
        "Recommendation"
    ]
].head(20)

,Rank,ID,Category,Experience Match,Education Match,Skills Match,Job Relevance,Overall Match,Recommendation
1,2,18448085,AUTOMOBILE,100.0,0.0,61.0,57.14,55.828,Moderate Match
4,5,12011623,ENGINEERING,60.0,0.0,85.0,57.14,57.428,Moderate Match
27,28,12351749,SALES,100.0,0.0,35.0,42.86,42.572,Weak Match
12,13,13264796,ENGINEERING,100.0,0.0,8.0,42.86,31.772,Weak Match
0,1,21156767,CONSULTANT,100.0,0.0,51.0,42.86,48.972,Weak Match
120,121,26585242,HEALTHCARE,100.0,0.0,16.0,28.57,32.114,Weak Match
93,94,34953092,BANKING,0.0,0.0,38.0,28.57,20.914,Weak Match
1887,1888,28998957,APPAREL,0.0,0.0,16.0,28.57,12.114,Weak Match
5,6,22946204,AUTOMOBILE,100.0,0.0,58.0,28.57,48.914,Weak Match
7,8,50328713,ENGINEERING,100.0,0.0,56.0,28.57,48.114,Weak Match


In [224]:
print("Job relevance terms:")
print(job_profile["job_relevance_terms"])

print("\nTop candidates by relevance:")
print(
    ranking_df.sort_values(
        "Job Relevance",
        ascending=False
    )[
        ["ID", "Category", "Job Relevance", "Overall Match"]
    ].head(10).to_string(index=False)
)

Job relevance terms:
['data science', 'data scientist', 'data analyst', 'data analysis', 'statistical modeling', 'machine learning', 'predictive modeling']

Top candidates by relevance:
      ID    Category  Job Relevance  Overall Match
18448085  AUTOMOBILE          57.14         55.828
12011623 ENGINEERING          57.14         57.428
12351749       SALES          42.86         42.572
13264796 ENGINEERING          42.86         31.772
21156767  CONSULTANT          42.86         48.972
26585242  HEALTHCARE          28.57         32.114
34953092     BANKING          28.57         20.914
28998957     APPAREL          28.57         12.114
22946204  AUTOMOBILE          28.57         48.914
50328713 ENGINEERING          28.57         48.114


In [225]:
def extract_job_relevance(resume_text, job_profile):

    text = str(resume_text).lower()

    relevance_terms = [
        term.lower().strip()
        for term in job_profile.get("job_relevance_terms", [])
    ]

    if not relevance_terms:
        return 0.0, []

    matched_terms = []

    for term in relevance_terms:

        if term in text:
            matched_terms.append(term)

    relevance_score = (
        len(matched_terms) / len(relevance_terms)
    ) * 100

    return round(relevance_score, 2), matched_terms

In [226]:
candidate_id = 21156767

resume_text = df.loc[
    df["ID"] == candidate_id,
    "Resume_str"
].iloc[0]

score, terms = extract_job_relevance(
    resume_text,
    job_profile
)

print("Candidate ID:", candidate_id)
print("Relevance Score:", score)
print("Matched Terms:", terms)

Candidate ID: 21156767
Relevance Score: 42.86
Matched Terms: ['data science', 'data analysis', 'machine learning']


In [227]:
screening_results = []

for _, row in df.iterrows():

    result = screen_candidate(
        row["Cleaned_Resume"],
        job_profile["required_skills"],
        job_profile["preferred_skills"],
        required_experience,
        required_education,
        job_profile
    )

    explanation = generate_explanation(
        result,
        required_experience
    )

    screening_results.append({
        "ID": row["ID"],
        "Category": row["Category"],
        "Experience Match": result["experience_match"],
        "Education Match": result["education_match"],
        "Skills Match": result["skills_match"],
        "Job Relevance": result["job_relevance"],
        "Overall Match": result["overall_match"],
        "Recommendation": get_recommendation(result["overall_match"]),
        "Matched Skills": ", ".join(
            result["skill_details"]["matched_required"]
        ),
        "Missing Skills": ", ".join(
            result["skill_details"]["missing_required"]
        ),
        "Relevant Terms": ", ".join(
            result["relevant_terms"]
        ),
        "Explanation": explanation
    })

ranking_df = pd.DataFrame(screening_results)

ranking_df = ranking_df.sort_values(
    "Overall Match",
    ascending=False
).reset_index(drop=True)

ranking_df.insert(
    0,
    "Rank",
    range(1, len(ranking_df) + 1)
)

print(
    ranking_df[
        [
            "Rank",
            "ID",
            "Category",
            "Experience Match",
            "Education Match",
            "Skills Match",
            "Job Relevance",
            "Overall Match",
            "Recommendation"
        ]
    ].head(10).to_string(index=False)
)

 Rank       ID               Category  Experience Match  Education Match  Skills Match  Job Relevance  Overall Match Recommendation
    1 21156767             CONSULTANT             100.0            100.0          51.0          42.86          71.83     Good Match
    2 18448085             AUTOMOBILE             100.0             60.0          61.0          57.14          69.97 Moderate Match
    3 12011623            ENGINEERING              60.0             60.0          85.0          57.14          69.57 Moderate Match
    4 29839396                BANKING             100.0            100.0          35.0          14.29          61.14 Moderate Match
    5 20566550            ENGINEERING             100.0            100.0          35.0          14.29          61.14 Moderate Match
    6 18067556 INFORMATION-TECHNOLOGY             100.0            100.0          31.0          14.29          59.54 Moderate Match
    7 11813872            AGRICULTURE             100.0            100.0    

In [228]:
print(
    ranking_df[
        [
            "Rank",
            "ID",
            "Overall Match",
            "Recommendation",
            "Matched Skills",
            "Missing Skills",
            "Relevant Terms",
            "Explanation"
        ]
    ].head(5).to_string(index=False)
)

 Rank       ID  Overall Match Recommendation                                                                                            Matched Skills                                                                                        Missing Skills                                                     Relevant Terms                                                                                                                                                                                                                                                                                                   Explanation
    1 21156767          71.83     Good Match                                          python, sql, data analysis, data visualization, machine learning                                          pandas, numpy, statistics, predictive modeling, scikit-learn                      data science, data analysis, machine learning          Meets the required experience with 6.0 years of e

In [229]:
print("===== SCORE STATISTICS =====")
print(ranking_df["Overall Match"].describe())

print("\n===== RECOMMENDATION DISTRIBUTION =====")
print(ranking_df["Recommendation"].value_counts())

===== SCORE STATISTICS =====
count    2481.000000
mean       21.305615
std        12.760402
min         0.000000
25%        20.000000
50%        25.000000
75%        25.000000
max        71.830000
Name: Overall Match, dtype: float64

===== RECOMMENDATION DISTRIBUTION =====
Recommendation
Weak Match        2443
Moderate Match      37
Good Match           1
Name: count, dtype: int64


In [230]:
top_id = ranking_df.iloc[0]["ID"]

top_resume = df.loc[
    df["ID"] == top_id,
    "Cleaned_Resume"
].iloc[0]

result = screen_candidate(
    top_resume,
    job_profile["required_skills"],
    job_profile["preferred_skills"],
    required_experience,
    required_education,
    job_profile
)

print("===== TOP CANDIDATE =====")
print("ID:", top_id)
print("Category:", ranking_df.iloc[0]["Category"])
print("Overall Match:", result["overall_match"])
print("Experience:", result["experience"])
print("Experience Match:", result["experience_match"])
print("Education Match:", result["education_match"])
print("Skills Match:", result["skills_match"])
print("Job Relevance:", result["job_relevance"])

print("\nMatched Skills:")
print(result["skill_details"]["matched_required"])

print("\nMissing Skills:")
print(result["skill_details"]["missing_required"])

print("\nRelevant Terms:")
print(result["relevant_terms"])

===== TOP CANDIDATE =====
ID: 21156767
Category: CONSULTANT
Overall Match: 71.83
Experience: 6.0
Experience Match: 100
Education Match: 100.0
Skills Match: 51.0
Job Relevance: 42.86

Matched Skills:
['python', 'sql', 'data analysis', 'data visualization', 'machine learning']

Missing Skills:
['pandas', 'numpy', 'statistics', 'predictive modeling', 'scikit-learn']

Relevant Terms:
['data science', 'data analysis', 'machine learning']


In [231]:
def generate_explanation(result, required_experience):

    if result["experience"] >= required_experience:
        experience_text = (
            f"Meets the required experience with "
            f"{result['experience']} years of experience."
        )
    else:
        experience_text = (
            f"Has {result['experience']} years of experience "
            f"against the required {required_experience} years."
        )

    if result["education_match"] == 100:
        education_text = "Education is highly aligned with the job requirements."
    elif result["education_match"] == 70:
        education_text = "Education is reasonably aligned with the job requirements."
    elif result["education_match"] == 20:
        education_text = "Has a degree, but the field is not closely aligned."
    else:
        education_text = "Required education was not detected."

    matched = result["skill_details"]["matched_required"]
    missing = result["skill_details"]["missing_required"]

    skill_text = f"Matches {len(matched)} required skills."

    if missing:
        skill_text += (
            " Missing key skills: "
            + ", ".join(missing)
            + "."
        )

    relevant_terms = result["relevant_terms"]

    if relevant_terms:
        relevance_text = (
            " Relevant terms found: "
            + ", ".join(relevant_terms)
            + "."
        )
    else:
        relevance_text = ""

    return (
        experience_text + " "
        + education_text + " "
        + skill_text
        + relevance_text
    )

In [232]:
explanations = []

for _, row in df.iterrows():

    result = screen_candidate(
        row["Cleaned_Resume"],
        job_profile["required_skills"],
        job_profile["preferred_skills"],
        required_experience,
        required_education,
        job_profile
    )

    explanations.append(
        generate_explanation(
            result,
            required_experience
        )
    )

ranking_df["Explanation"] = explanations

print(
    ranking_df[
        [
            "Rank",
            "ID",
            "Category",
            "Overall Match",
            "Recommendation",
            "Explanation"
        ]
    ].head(5).to_string(index=False)
)

 Rank       ID    Category  Overall Match Recommendation                                                                                                                                                                                                                                                                                Explanation
    1 21156767  CONSULTANT          71.83     Good Match Meets the required experience with 15.0 years of experience. Required education was not detected. Matches 2 required skills. Missing key skills: python, sql, pandas, numpy, data visualization, machine learning, predictive modeling, scikit-learn. Relevant terms found: data analysis.
    2 18448085  AUTOMOBILE          69.97 Moderate Match             Meets the required experience with 11 years of experience. Required education was not detected. Matches 0 required skills. Missing key skills: python, sql, pandas, numpy, data analysis, statistics, data visualization, machine learning, predictive mode

In [233]:
ranking_df.to_csv(
    "final_candidate_ranking.csv",
    index=False
)

print("Saved: final_candidate_ranking.csv")
print("Rows:", len(ranking_df))
print("Columns:", ranking_df.columns.tolist())

Saved: final_candidate_ranking.csv
Rows: 2481
Columns: ['Rank', 'ID', 'Category', 'Experience Match', 'Education Match', 'Skills Match', 'Job Relevance', 'Overall Match', 'Recommendation', 'Matched Skills', 'Missing Skills', 'Relevant Terms', 'Explanation']


In [234]:
print("===== FINAL SCORE STATISTICS =====")
print(ranking_df["Overall Match"].describe())

print("\n===== FINAL RECOMMENDATION DISTRIBUTION =====")
print(ranking_df["Recommendation"].value_counts())

===== FINAL SCORE STATISTICS =====
count    2481.000000
mean       21.305615
std        12.760402
min         0.000000
25%        20.000000
50%        25.000000
75%        25.000000
max        71.830000
Name: Overall Match, dtype: float64

===== FINAL RECOMMENDATION DISTRIBUTION =====
Recommendation
Weak Match        2443
Moderate Match      37
Good Match           1
Name: count, dtype: int64


In [235]:
# Check the scoring columns
print(ranking_df.columns.tolist())

# Show top candidates with all score components
display(
    ranking_df.sort_values("Overall Match", ascending=False)
    .head(10)
)

['Rank', 'ID', 'Category', 'Experience Match', 'Education Match', 'Skills Match', 'Job Relevance', 'Overall Match', 'Recommendation', 'Matched Skills', 'Missing Skills', 'Relevant Terms', 'Explanation']


,Rank,ID,Category,Experience Match,Education Match,Skills Match,Job Relevance,Overall Match,Recommendation,Matched Skills,Missing Skills,Relevant Terms,Explanation
0,1,21156767,CONSULTANT,100.0,100.0,51.0,42.86,71.83,Good Match,"python, sql, data analysis, data visualization, machine learning","pandas, numpy, statistics, predictive modeling, scikit-learn","data science, data analysis, machine learning",Meets the required experience with 15.0 years of experience. Required education was not detected...
1,2,18448085,AUTOMOBILE,100.0,60.0,61.0,57.14,69.97,Moderate Match,"python, sql, pandas, numpy, data analysis, statistics, data visualization","machine learning, predictive modeling, scikit-learn","data science, data analyst, data analysis, statistical modeling",Meets the required experience with 11 years of experience. Required education was not detected. ...
2,3,12011623,ENGINEERING,60.0,60.0,85.0,57.14,69.57,Moderate Match,"python, sql, pandas, data analysis, statistics, data visualization, machine learning, predictive...","numpy, scikit-learn","data analyst, data analysis, machine learning, predictive modeling",Meets the required experience with 20.0 years of experience. Required education was not detected...
3,4,29839396,BANKING,100.0,100.0,35.0,14.29,61.14,Moderate Match,"python, data analysis, statistics","sql, pandas, numpy, data visualization, machine learning, predictive modeling, scikit-learn",data analysis,Meets the required experience with 20.0 years of experience. Required education was not detected...
4,5,20566550,ENGINEERING,100.0,100.0,35.0,14.29,61.14,Moderate Match,"python, sql, statistics","pandas, numpy, data analysis, data visualization, machine learning, predictive modeling, scikit-...",data science,Meets the required experience with 11.58 years of experience. Required education was not detecte...
6,7,11813872,AGRICULTURE,100.0,100.0,31.0,14.29,59.54,Moderate Match,"python, sql, pandas","numpy, data analysis, statistics, data visualization, machine learning, predictive modeling, sci...",data science,Meets the required experience with 19 years of experience. Required education was not detected. ...
5,6,18067556,INFORMATION-TECHNOLOGY,100.0,100.0,31.0,14.29,59.54,Moderate Match,"python, sql, data analysis","pandas, numpy, statistics, data visualization, machine learning, predictive modeling, scikit-learn",data analysis,Has 0.0 years of experience against the required 5 years. Required education was not detected. M...
7,8,11890896,ENGINEERING,100.0,100.0,28.0,14.29,58.34,Moderate Match,"sql, data analysis, statistics","python, pandas, numpy, data visualization, machine learning, predictive modeling, scikit-learn",data analysis,Meets the required experience with 10 years of experience. Required education was not detected. ...
9,10,22232367,AVIATION,100.0,100.0,23.0,14.29,56.34,Moderate Match,"python, sql","pandas, numpy, data analysis, statistics, data visualization, machine learning, predictive model...",data analyst,Meets the required experience with 10 years of experience. Required education was not detected. ...
8,9,12635195,INFORMATION-TECHNOLOGY,100.0,100.0,23.0,14.29,56.34,Moderate Match,"python, data analysis","sql, pandas, numpy, statistics, data visualization, machine learning, predictive modeling, sciki...",data analysis,Meets the required experience with 15.0 years of experience. Required education was not detected...


In [236]:
print(ranking_df["Overall Match"].describe())

# Check correlation between individual components and final score
score_cols = [
    col for col in ranking_df.columns
    if any(x in col.lower() for x in [
        "match", "score", "similarity", "education",
        "skill", "experience", "category"
    ])
]

print("Score-related columns:")
print(score_cols)

count    2481.000000
mean       21.305615
std        12.760402
min         0.000000
25%        20.000000
50%        25.000000
75%        25.000000
max        71.830000
Name: Overall Match, dtype: float64
Score-related columns:
['Category', 'Experience Match', 'Education Match', 'Skills Match', 'Overall Match', 'Matched Skills', 'Missing Skills']


In [237]:
print([name for name in globals() if callable(globals()[name])])

['get_ipython', 'exit', 'quit', 'open', 'datetime', 'extract_experience', 'extract_education', 'calculate_education_match', 'detailed_skill_match', 'calculate_final_skill_score', 'extract_job_relevance', 'screen_candidate', 'get_recommendation', 'explain_candidate', 'generate_explanation', 'value', 'show_candidate', 'get_eligibility', 'normalize_education_fields']


In [238]:
print(job_profile["required_skills"])
print(job_profile["preferred_skills"])

['python', 'sql', 'pandas', 'numpy', 'data analysis', 'statistics', 'data visualization', 'machine learning', 'predictive modeling', 'scikit-learn']
[]


In [239]:
test_cases = [
    ["python", "sql", "pandas"],
    ["python", "machine learning", "predictive modeling"],
    ["python", "sql", "pandas", "statistics",
     "machine learning", "predictive modeling", "scikit-learn"]
]

for skills in test_cases:

    skill_details = {
        "matched_required": skills,
        "missing_required": [],
        "matched_preferred": [],
        "required_score": 0,
        "preferred_score": 0
    }

    score = calculate_final_skill_score(skill_details)

    print(f"{skills}")
    print(f"Weighted Skill Score: {score}%")
    print("-" * 60)

['python', 'sql', 'pandas']
Weighted Skill Score: 31.0%
------------------------------------------------------------
['python', 'machine learning', 'predictive modeling']
Weighted Skill Score: 44.0%
------------------------------------------------------------
['python', 'sql', 'pandas', 'statistics', 'machine learning', 'predictive modeling', 'scikit-learn']
Weighted Skill Score: 82.0%
------------------------------------------------------------


In [240]:
print("Required experience:", required_experience)
print("Required education:", required_education)
print("Job profile:", job_profile)

Required experience: 5
Required education: ['data science', 'statistics', 'mathematics', 'computer science', 'information technology']
Job profile: {'job_title': 'Data Scientist', 'domain': 'Data Science', 'required_skills': ['python', 'sql', 'pandas', 'numpy', 'data analysis', 'statistics', 'data visualization', 'machine learning', 'predictive modeling', 'scikit-learn'], 'preferred_skills': [], 'job_relevance_terms': ['data science', 'data scientist', 'data analyst', 'data analysis', 'statistical modeling', 'machine learning', 'predictive modeling']}


In [241]:
print("===== TOP 15 EDUCATION EXTRACTION =====")

for candidate_id in ranking_df.head(15)["ID"]:

    resume_text = df.loc[
        df["ID"] == candidate_id,
        "Resume_str"
    ].iloc[0]

    education = extract_education(resume_text)

    print("\nID:", candidate_id)
    print("Education:", education)

===== TOP 15 EDUCATION EXTRACTION =====

ID: 21156767
Education: {'degrees': ['b.e', 'm.sc'], 'fields': ['data science', 'computer science', 'machine learning']}

ID: 18448085
Education: {'degrees': ['b.e', 'm.sc'], 'fields': ['mechanical engineering', 'industrial and systems engineering']}

ID: 12011623
Education: {'degrees': [], 'fields': ['machine learning']}

ID: 29839396
Education: {'degrees': ['b.sc'], 'fields': ['statistics', 'finance']}

ID: 20566550
Education: {'degrees': ['b.e', 'b.sc'], 'fields': ['statistics', 'mathematics', 'mechanical engineering']}

ID: 18067556
Education: {'degrees': [], 'fields': ['information technology', 'finance', 'marketing']}

ID: 11813872
Education: {'degrees': ['m.sc'], 'fields': ['computer science', 'finance']}

ID: 11890896
Education: {'degrees': ['b.sc'], 'fields': ['statistics', 'mathematics', 'business administration']}

ID: 12635195
Education: {'degrees': ['b.sc', 'm.sc'], 'fields': ['information technology', 'information systems']}

ID: 2

In [251]:
def show_candidate(candidate_id):

    row = ranking_df[
        ranking_df["ID"] == candidate_id
    ].iloc[0]

    print("=" * 70)
    print(f"CANDIDATE ID: {candidate_id}")
    print("=" * 70)

    print(f"Rank:              {row['Rank']}")
    print(f"Category:          {row['Category']}")
    print(f"Experience:        {row['Experience']} years")

    print("\n--- SCORE BREAKDOWN ---")

    print(f"Experience Match:  {row['Experience Match']}%")
    print(f"Education Match:   {row['Education Match']}%")
    print(f"Skills Match:       {row['Skills Match']}%")
    print(f"Job Relevance:      {row['Job Relevance']}%")
    print(f"Overall Match:      {row['Overall Match']}%")

    print("\n--- RECOMMENDATION ---")
    print(row["Recommendation"])

    print("\n--- MATCHED SKILLS ---")
    print(row["Matched Skills"])

    print("\n--- MISSING SKILLS ---")
    print(row["Missing Skills"])

    print("\n--- RELEVANT TERMS ---")
    print(row["Relevant Terms"])

    print("\n--- EXPLANATION ---")
    print(row["Explanation"])

In [255]:
ranking_df["Education Match"] = df["Education_Match_New"].values

In [256]:
display(
    ranking_df[
        [
            "ID",
            "Experience Match",
            "Education Match",
            "Skills Match",
            "Job Relevance",
            "Overall Match",
            "Recommendation"
        ]
    ].head(10)
)

,ID,Experience Match,Education Match,Skills Match,Job Relevance,Overall Match,Recommendation
0,21156767,100.0,0.0,51.0,42.86,71.83,Good Match
1,18448085,100.0,0.0,61.0,57.14,69.97,Moderate Match
2,12011623,60.0,0.0,85.0,57.14,69.57,Moderate Match
3,29839396,100.0,0.0,35.0,14.29,61.14,Moderate Match
4,20566550,100.0,0.0,35.0,14.29,61.14,Moderate Match
5,18067556,100.0,0.0,31.0,14.29,59.54,Moderate Match
6,11813872,100.0,0.0,31.0,14.29,59.54,Moderate Match
7,11890896,100.0,0.0,28.0,14.29,58.34,Moderate Match
8,12635195,100.0,0.0,23.0,14.29,56.34,Moderate Match
9,22232367,100.0,0.0,23.0,14.29,56.34,Moderate Match


In [265]:
ranking_df["Overall Match"] = (
    ranking_df["Experience Match"] * 0.20 +
    ranking_df["Education Match"] * 0.20 +
    ranking_df["Skills Match"] * 0.40 +
    ranking_df["Job Relevance"] * 0.20
).round(2)

In [269]:
# Keep only eligible candidates
eligible_df = (
    ranking_df[
        ranking_df["Eligibility"] == "Eligible"
    ]
    .sort_values(
        ["Overall Match", "Skills Match", "Job Relevance"],
        ascending=False
    )
    .reset_index(drop=True)
)

eligible_df["Eligible Rank"] = eligible_df.index + 1

print("Eligible candidates:", len(eligible_df))

display(
    eligible_df.head(20)[
        [
            "Eligible Rank",
            "ID",
            "Category",
            "Experience Match",
            "Education Match",
            "Skills Match",
            "Job Relevance",
            "Overall Match",
            "Recommendation",
            "Matched Skills",
            "Missing Skills",
            "Relevant Terms"
        ]
    ]
)

Eligible candidates: 158


,Eligible Rank,ID,Category,Experience Match,Education Match,Skills Match,Job Relevance,Overall Match,Recommendation,Matched Skills,Missing Skills,Relevant Terms
0,1,20674668,INFORMATION-TECHNOLOGY,100.0,100.0,35.0,0.00,54.00,Moderate Match,"python, sql, statistics","pandas, numpy, data analysis, data visualization, machine learning, predictive modeling, scikit-...",
1,2,11332602,FITNESS,100.0,100.0,20.0,14.29,50.86,Moderate Match,"data analysis, statistics","python, sql, pandas, numpy, data visualization, machine learning, predictive modeling, scikit-learn",data analysis
2,3,86549455,FINANCE,100.0,100.0,8.0,14.29,46.06,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin...",data analysis
3,4,27847081,HR,100.0,100.0,8.0,14.29,46.06,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin...",data analysis
4,5,15941675,ENGINEERING,100.0,100.0,8.0,14.29,46.06,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin...",data analysis
5,6,16121387,HEALTHCARE,100.0,100.0,8.0,14.29,46.06,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin...",data analysis
6,7,91539554,DIGITAL-MEDIA,100.0,100.0,8.0,14.29,46.06,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin...",data analysis
7,8,17412079,HR,100.0,100.0,8.0,14.29,46.06,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin...",data analysis
8,9,41910253,HEALTHCARE,100.0,100.0,8.0,14.29,46.06,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin...",data analysis
9,10,11432686,CHEF,100.0,100.0,8.0,14.29,46.06,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin...",data analysis


In [270]:
top_resume = eligible_df.head(10).copy()

display(
    top_resume[
        [
            "Eligible Rank",
            "ID",
            "Category",
            "Experience Match",
            "Education Match",
            "Skills Match",
            "Job Relevance",
            "Overall Match",
            "Recommendation",
            "Matched Skills",
            "Missing Skills",
            "Relevant Terms"
        ]
    ]
)

,Eligible Rank,ID,Category,Experience Match,Education Match,Skills Match,Job Relevance,Overall Match,Recommendation,Matched Skills,Missing Skills,Relevant Terms
0,1,20674668,INFORMATION-TECHNOLOGY,100.0,100.0,35.0,0.00,54.00,Moderate Match,"python, sql, statistics","pandas, numpy, data analysis, data visualization, machine learning, predictive modeling, scikit-...",
1,2,11332602,FITNESS,100.0,100.0,20.0,14.29,50.86,Moderate Match,"data analysis, statistics","python, sql, pandas, numpy, data visualization, machine learning, predictive modeling, scikit-learn",data analysis
2,3,86549455,FINANCE,100.0,100.0,8.0,14.29,46.06,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin...",data analysis
3,4,27847081,HR,100.0,100.0,8.0,14.29,46.06,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin...",data analysis
4,5,15941675,ENGINEERING,100.0,100.0,8.0,14.29,46.06,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin...",data analysis
5,6,16121387,HEALTHCARE,100.0,100.0,8.0,14.29,46.06,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin...",data analysis
6,7,91539554,DIGITAL-MEDIA,100.0,100.0,8.0,14.29,46.06,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin...",data analysis
7,8,17412079,HR,100.0,100.0,8.0,14.29,46.06,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin...",data analysis
8,9,41910253,HEALTHCARE,100.0,100.0,8.0,14.29,46.06,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin...",data analysis
9,10,11432686,CHEF,100.0,100.0,8.0,14.29,46.06,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin...",data analysis


In [271]:
best_candidate = eligible_df.iloc[0]

print("=" * 75)
print("BEST ELIGIBLE CANDIDATE")
print("=" * 75)

print(f"Eligible Rank:      {best_candidate['Eligible Rank']}")
print(f"ID:                 {best_candidate['ID']}")
print(f"Category:           {best_candidate['Category']}")
print(f"Experience Match:   {best_candidate['Experience Match']}%")
print(f"Education Match:    {best_candidate['Education Match']}%")
print(f"Skills Match:       {best_candidate['Skills Match']}%")
print(f"Job Relevance:      {best_candidate['Job Relevance']}%")
print(f"Overall Match:      {best_candidate['Overall Match']}%")
print(f"Recommendation:     {best_candidate['Recommendation']}")
print(f"Matched Skills:     {best_candidate['Matched Skills']}")
print(f"Missing Skills:     {best_candidate['Missing Skills']}")
print(f"Relevant Terms:     {best_candidate['Relevant Terms']}")
print(f"Explanation:        {best_candidate['Explanation']}")

BEST ELIGIBLE CANDIDATE
Eligible Rank:      1
ID:                 20674668
Category:           INFORMATION-TECHNOLOGY
Experience Match:   100.0%
Education Match:    100.0%
Skills Match:       35.0%
Job Relevance:      0.0%
Overall Match:      54.0%
Recommendation:     Moderate Match
Matched Skills:     python, sql, statistics
Missing Skills:     pandas, numpy, data analysis, data visualization, machine learning, predictive modeling, scikit-learn
Relevant Terms:     
Explanation:        Meets the required experience with 7.0 years of experience. Education is highly aligned with the job requirements. Matches 0 required skills. Missing key skills: python, sql, pandas, numpy, data analysis, statistics, data visualization, machine learning, predictive modeling, scikit-learn.


In [288]:
# Create new ranking score
ranking_df["Screening Score"] = (
    0.40 * ranking_df["Skills Match"] +
    0.30 * ranking_df["Job Relevance"] +
    0.20 * ranking_df["Education Match"] +
    0.10 * ranking_df["Experience Match"]
)

print("Screening Score created successfully.")
print(ranking_df["Screening Score"].describe())

Screening Score created successfully.
count    2481.000000
mean       10.333280
std         8.113463
min         0.000000
25%         8.000000
50%        10.000000
75%        10.000000
max        57.142000
Name: Screening Score, dtype: float64


In [275]:
eligible_df = (
    ranking_df[
        ranking_df["Eligibility"] == "Eligible"
    ]
    .sort_values(
        "Screening Score",
        ascending=False
    )
    .reset_index(drop=True)
)

eligible_df["Eligible Rank"] = eligible_df.index + 1

print("Eligible candidates:", len(eligible_df))

Eligible candidates: 158


In [276]:
display(
    eligible_df.head(15)[
        [
            "Eligible Rank",
            "ID",
            "Category",
            "Experience Match",
            "Education Match",
            "Skills Match",
            "Job Relevance",
            "Screening Score",
            "Recommendation",
            "Matched Skills",
            "Missing Skills",
            "Relevant Terms"
        ]
    ]
)

,Eligible Rank,ID,Category,Experience Match,Education Match,Skills Match,Job Relevance,Screening Score,Recommendation,Matched Skills,Missing Skills,Relevant Terms
0,1,20674668,INFORMATION-TECHNOLOGY,100.0,100.0,35.0,0.00,44.000,Moderate Match,"python, sql, statistics","pandas, numpy, data analysis, data visualization, machine learning, predictive modeling, scikit-...",
1,2,11332602,FITNESS,100.0,100.0,20.0,14.29,42.287,Moderate Match,"data analysis, statistics","python, sql, pandas, numpy, data visualization, machine learning, predictive modeling, scikit-learn",data analysis
2,3,86549455,FINANCE,100.0,100.0,8.0,14.29,37.487,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin...",data analysis
3,4,27847081,HR,100.0,100.0,8.0,14.29,37.487,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin...",data analysis
4,5,15941675,ENGINEERING,100.0,100.0,8.0,14.29,37.487,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin...",data analysis
5,6,16121387,HEALTHCARE,100.0,100.0,8.0,14.29,37.487,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin...",data analysis
6,7,91539554,DIGITAL-MEDIA,100.0,100.0,8.0,14.29,37.487,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin...",data analysis
7,8,17412079,HR,100.0,100.0,8.0,14.29,37.487,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin...",data analysis
8,9,41910253,HEALTHCARE,100.0,100.0,8.0,14.29,37.487,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin...",data analysis
9,10,11432686,CHEF,100.0,100.0,8.0,14.29,37.487,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin...",data analysis


In [277]:
best_candidate = eligible_df.iloc[0]

print("=" * 75)
print("BEST MATCHING CANDIDATE")
print("=" * 75)

print(f"ID:                 {best_candidate['ID']}")
print(f"Category:           {best_candidate['Category']}")
print(f"Experience Match:   {best_candidate['Experience Match']}%")
print(f"Education Match:    {best_candidate['Education Match']}%")
print(f"Skills Match:       {best_candidate['Skills Match']}%")
print(f"Job Relevance:      {best_candidate['Job Relevance']}%")
print(f"Screening Score:    {best_candidate['Screening Score']:.2f}%")
print(f"Recommendation:     {best_candidate['Recommendation']}")
print(f"Matched Skills:     {best_candidate['Matched Skills']}")
print(f"Missing Skills:     {best_candidate['Missing Skills']}")
print(f"Relevant Terms:     {best_candidate['Relevant Terms']}")

BEST MATCHING CANDIDATE
ID:                 20674668
Category:           INFORMATION-TECHNOLOGY
Experience Match:   100.0%
Education Match:    100.0%
Skills Match:       35.0%
Job Relevance:      0.0%
Screening Score:    44.00%
Recommendation:     Moderate Match
Matched Skills:     python, sql, statistics
Missing Skills:     pandas, numpy, data analysis, data visualization, machine learning, predictive modeling, scikit-learn
Relevant Terms:     


In [ ]:
def extract_education(resume_text):
    text = str(resume_text).lower()

    # ---------------------------------------------------------
    # 1. Try to isolate the Education section
    # ---------------------------------------------------------
    education_section = text

    education_patterns = [
        r'\beducation\b(.*?)(?=\b(?:skills|experience|work experience|'
        r'professional experience|employment|projects|certifications|'
        r'achievements|summary|objective)\b)',
        r'\bqualification[s]?\b(.*?)(?=\b(?:skills|experience|work experience|'
        r'employment|projects|certifications)\b)'
    ]

    for pattern in education_patterns:
        match = re.search(pattern, text, re.DOTALL)

        if match:
            education_section = match.group(1)
            break

    # ---------------------------------------------------------
    # 2. Extract degrees
    # ---------------------------------------------------------
    degrees = []

    degree_patterns = {
    "b.e": r'\b(?:b\.e\.?|bachelors?\s+of\s+engineering)\b',

    "b.tech": r'\b(?:b\.tech\.?|btech|bachelors?\s+of\s+technology)\b',

    "b.sc": r'\b(?:b\.sc\.?|bsc|b\.s\.?|b\.s|'
             r'bachelors?\s+of\s+science)\b',

    "m.e": r'\b(?:m\.e\.?|masters?\s+of\s+engineering)\b',

    "m.tech": r'\b(?:m\.tech\.?|mtech|masters?\s+of\s+technology)\b',

    "m.sc": r'\b(?:m\.sc\.?|msc|m\.s\.?|m\.s|'
             r'masters?\s+of\s+science)\b',

    "mba": r'\bmba\b',

    "ph.d": r'\b(?:ph\.d\.?|phd|doctor\s+of\s+philosophy)\b'
}

    for degree, pattern in degree_patterns.items():
        if re.search(pattern, education_section):
            degrees.append(degree)

    # ---------------------------------------------------------
    # 3. Extract fields
    # ---------------------------------------------------------
    field_patterns = [
        "data science",
        "computer science",
        "information technology",
        "information systems",
        "statistics",
        "mathematics",
        "machine learning",
        "artificial intelligence",
        "software engineering",
        "computer engineering",
        "mechanical engineering",
        "electrical engineering",
        "civil engineering",
        "business administration",
        "finance",
        "marketing",
        "industrial and systems engineering"
    ]

    fields = []

    for field in field_patterns:
        if field in education_section:
            fields.append(field)

    return {
        "degrees": degrees,
        "fields": fields
    }

In [ ]:
for candidate_id in [12011623, 10265057]:

    resume = df.loc[
        df["ID"] == candidate_id,
        "Resume_str"
    ].iloc[0]

    print("=" * 60)
    print("ID:", candidate_id)
    print(extract_education(resume))

In [ ]:
education_scores = []

for _, row in df.iterrows():

    education = extract_education(row["Resume_str"])

    score = calculate_education_match(
        education,
        required_education
    )

    education_scores.append(score)

df["Education_Match_New"] = education_scores

In [ ]:
print("===== NEW EDUCATION DISTRIBUTION =====")
print(df["Education_Match_New"].value_counts().sort_index())

print("\n===== NEW EDUCATION STATISTICS =====")
print(df["Education_Match_New"].describe())
print("\nEducation Match >= 60%:",
      (df["Education_Match_New"] >= 60).sum())

print("Education Match = 100%:",
      (df["Education_Match_New"] == 100).sum())
education_check = (
    df[
        ["ID", "Category", "Education_Match_New"]
    ]
    .sort_values("Education_Match_New", ascending=False)
)

display(education_check.head(20))

In [ ]:
ranking_df["Education Match"] = df["Education_Match_New"]
ranking_df["Overall Match"] = (
    ranking_df["Experience Match"] * 0.20 +
    ranking_df["Education Match"] * 0.20 +
    ranking_df["Skills Match"] * 0.40 +
    ranking_df["Job Relevance"] * 0.20
)

In [278]:
def get_eligibility(row):

    experience_gap = row["Experience Match"] < 100
    education_gap = row["Education Match"] < 100

    if not experience_gap and not education_gap:
        return "Eligible"

    elif experience_gap and education_gap:
        return "Experience + Education Gap"

    elif experience_gap:
        return "Experience Gap"

    else:
        return "Education Gap"


ranking_df["Eligibility"] = ranking_df.apply(
    get_eligibility,
    axis=1
)

In [279]:
print("===== ELIGIBILITY DISTRIBUTION =====")

print(
    ranking_df["Eligibility"].value_counts()
)

===== ELIGIBILITY DISTRIBUTION =====
Eligibility
Education Gap                 1590
Experience + Education Gap     677
Eligible                       158
Experience Gap                  56
Name: count, dtype: int64


In [280]:
display(
    eligible_df.head(15)[
        [
            "Eligible Rank",
            "ID",
            "Category",
            "Experience Match",
            "Education Match",
            "Skills Match",
            "Job Relevance",
            "Overall Match",
            "Screening Score",
            "Recommendation",
            "Matched Skills",
            "Missing Skills",
            "Relevant Terms"
        ]
    ]
)

,Eligible Rank,ID,Category,Experience Match,Education Match,Skills Match,Job Relevance,Overall Match,Screening Score,Recommendation,Matched Skills,Missing Skills,Relevant Terms
0,1,20674668,INFORMATION-TECHNOLOGY,100.0,100.0,35.0,0.00,54.00,44.000,Moderate Match,"python, sql, statistics","pandas, numpy, data analysis, data visualization, machine learning, predictive modeling, scikit-...",
1,2,11332602,FITNESS,100.0,100.0,20.0,14.29,50.86,42.287,Moderate Match,"data analysis, statistics","python, sql, pandas, numpy, data visualization, machine learning, predictive modeling, scikit-learn",data analysis
2,3,86549455,FINANCE,100.0,100.0,8.0,14.29,46.06,37.487,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin...",data analysis
3,4,27847081,HR,100.0,100.0,8.0,14.29,46.06,37.487,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin...",data analysis
4,5,15941675,ENGINEERING,100.0,100.0,8.0,14.29,46.06,37.487,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin...",data analysis
5,6,16121387,HEALTHCARE,100.0,100.0,8.0,14.29,46.06,37.487,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin...",data analysis
6,7,91539554,DIGITAL-MEDIA,100.0,100.0,8.0,14.29,46.06,37.487,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin...",data analysis
7,8,17412079,HR,100.0,100.0,8.0,14.29,46.06,37.487,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin...",data analysis
8,9,41910253,HEALTHCARE,100.0,100.0,8.0,14.29,46.06,37.487,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin...",data analysis
9,10,11432686,CHEF,100.0,100.0,8.0,14.29,46.06,37.487,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin...",data analysis


In [281]:
eligible_df = (
    ranking_df[
        ranking_df["Eligibility"] == "Eligible"
    ]
    .sort_values("Overall Match", ascending=False)
    .reset_index(drop=True)
)

eligible_df["Eligible Rank"] = eligible_df.index + 1

print("Eligible candidates:", len(eligible_df))

display(
    eligible_df.head(15)[
        [
            "Eligible Rank",
            "ID",
            "Category",
            "Experience Match",
            "Education Match",
            "Skills Match",
            "Job Relevance",
            "Overall Match",
            "Screening Score",
            "Recommendation",
            "Matched Skills",
            "Missing Skills",
            "Relevant Terms"
        ]
    ]
)

Eligible candidates: 158


,Eligible Rank,ID,Category,Experience Match,Education Match,Skills Match,Job Relevance,Overall Match,Screening Score,Recommendation,Matched Skills,Missing Skills,Relevant Terms
0,1,20674668,INFORMATION-TECHNOLOGY,100.0,100.0,35.0,0.00,54.00,44.000,Moderate Match,"python, sql, statistics","pandas, numpy, data analysis, data visualization, machine learning, predictive modeling, scikit-...",
1,2,11332602,FITNESS,100.0,100.0,20.0,14.29,50.86,42.287,Moderate Match,"data analysis, statistics","python, sql, pandas, numpy, data visualization, machine learning, predictive modeling, scikit-learn",data analysis
2,3,86549455,FINANCE,100.0,100.0,8.0,14.29,46.06,37.487,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin...",data analysis
3,4,27847081,HR,100.0,100.0,8.0,14.29,46.06,37.487,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin...",data analysis
4,5,15941675,ENGINEERING,100.0,100.0,8.0,14.29,46.06,37.487,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin...",data analysis
5,6,16121387,HEALTHCARE,100.0,100.0,8.0,14.29,46.06,37.487,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin...",data analysis
6,7,91539554,DIGITAL-MEDIA,100.0,100.0,8.0,14.29,46.06,37.487,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin...",data analysis
7,8,17412079,HR,100.0,100.0,8.0,14.29,46.06,37.487,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin...",data analysis
8,9,41910253,HEALTHCARE,100.0,100.0,8.0,14.29,46.06,37.487,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin...",data analysis
9,10,11432686,CHEF,100.0,100.0,8.0,14.29,46.06,37.487,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin...",data analysis


In [282]:
best_candidate = eligible_df.iloc[0]

print("=" * 75)
print("BEST ELIGIBLE CANDIDATE")
print("=" * 75)

for col in [
    "Eligible Rank",
    "ID",
    "Category",
    "Experience Match",
    "Education Match",
    "Skills Match",
    "Job Relevance",
    "Overall Match",
    "Screening Score",
    "Recommendation",
    "Matched Skills",
    "Missing Skills",
    "Relevant Terms"
]:
    print(f"{col}: {best_candidate[col]}")

BEST ELIGIBLE CANDIDATE
Eligible Rank: 1
ID: 20674668
Category: INFORMATION-TECHNOLOGY
Experience Match: 100.0
Education Match: 100.0
Skills Match: 35.0
Job Relevance: 0.0
Overall Match: 54.0
Screening Score: 44.0
Recommendation: Moderate Match
Matched Skills: python, sql, statistics
Missing Skills: pandas, numpy, data analysis, data visualization, machine learning, predictive modeling, scikit-learn
Relevant Terms: 


In [283]:
print("Ranking columns:")
print(ranking_df.columns.tolist())

print("\nCandidate 20674668:")
display(
    ranking_df[
        ranking_df["ID"] == 20674668
    ].T
)

Ranking columns:
['Rank', 'ID', 'Category', 'Experience Match', 'Education Match', 'Skills Match', 'Job Relevance', 'Overall Match', 'Recommendation', 'Matched Skills', 'Missing Skills', 'Relevant Terms', 'Explanation', 'Eligibility', 'Screening Score']

Candidate 20674668:


,2
Rank,3
ID,20674668
Category,INFORMATION-TECHNOLOGY
Experience Match,100.0
Education Match,100.0
Skills Match,35.0
Job Relevance,0.0
Overall Match,54.0
Recommendation,Moderate Match
Matched Skills,"python, sql, statistics"


In [284]:
print("Current Screening Score formula values:")

row = ranking_df[ranking_df["ID"] == 20674668].iloc[0]

print("Experience Match:", row["Experience Match"])
print("Education Match:", row["Education Match"])
print("Skills Match:", row["Skills Match"])
print("Job Relevance:", row["Job Relevance"])
print("Overall Match:", row["Overall Match"])
print("Screening Score:", row["Screening Score"])

Current Screening Score formula values:
Experience Match: 100.0
Education Match: 100.0
Skills Match: 35.0
Job Relevance: 0.0
Overall Match: 54.0
Screening Score: 44.0


In [295]:
ranking_df["Screening Score"] = (
    ranking_df["Experience Match"] * 0.20 +
    ranking_df["Education Match"] * 0.20 +
    ranking_df["Skills Match"] * 0.40 +
    ranking_df["Job Relevance"] * 0.20
).round(2)

In [290]:
ranking_df["Overall Match"] = ranking_df["Screening Score"]

In [296]:
def get_recommendation(score):
    if score >= 70:
        return "Good Match"
    elif score >= 50:
        return "Moderate Match"
    else:
        return "Weak Match"

ranking_df["Recommendation"] = ranking_df["Screening Score"].apply(
    get_recommendation
)

In [297]:
eligible_df = (
    ranking_df[ranking_df["Eligibility"] == "Eligible"]
    .sort_values("Screening Score", ascending=False)
    .reset_index(drop=True)
)

eligible_df["Eligible Rank"] = eligible_df.index + 1

In [298]:
display(
    eligible_df.head(15)[
        [
            "Eligible Rank",
            "ID",
            "Category",
            "Experience Match",
            "Education Match",
            "Skills Match",
            "Job Relevance",
            "Screening Score",
            "Recommendation",
            "Matched Skills",
            "Missing Skills",
            "Relevant Terms"
        ]
    ]
)

,Eligible Rank,ID,Category,Experience Match,Education Match,Skills Match,Job Relevance,Screening Score,Recommendation,Matched Skills,Missing Skills,Relevant Terms
0,1,20674668,INFORMATION-TECHNOLOGY,100.0,100.0,35.0,0.00,54.00,Moderate Match,"python, sql, statistics","pandas, numpy, data analysis, data visualization, machine learning, predictive modeling, scikit-...",
1,2,11332602,FITNESS,100.0,100.0,20.0,14.29,50.86,Moderate Match,"data analysis, statistics","python, sql, pandas, numpy, data visualization, machine learning, predictive modeling, scikit-learn",data analysis
2,3,86549455,FINANCE,100.0,100.0,8.0,14.29,46.06,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin...",data analysis
3,4,27847081,HR,100.0,100.0,8.0,14.29,46.06,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin...",data analysis
4,5,15941675,ENGINEERING,100.0,100.0,8.0,14.29,46.06,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin...",data analysis
5,6,16121387,HEALTHCARE,100.0,100.0,8.0,14.29,46.06,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin...",data analysis
6,7,91539554,DIGITAL-MEDIA,100.0,100.0,8.0,14.29,46.06,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin...",data analysis
7,8,17412079,HR,100.0,100.0,8.0,14.29,46.06,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin...",data analysis
8,9,41910253,HEALTHCARE,100.0,100.0,8.0,14.29,46.06,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin...",data analysis
9,10,11432686,CHEF,100.0,100.0,8.0,14.29,46.06,Weak Match,data analysis,"python, sql, pandas, numpy, statistics, data visualization, machine learning, predictive modelin...",data analysis


In [294]:
best_candidate = eligible_df.iloc[0]

print("=" * 75)
print("BEST ELIGIBLE CANDIDATE")
print("=" * 75)

print(f"Eligible Rank:      {best_candidate['Eligible Rank']}")
print(f"ID:                 {best_candidate['ID']}")
print(f"Category:           {best_candidate['Category']}")
print(f"Experience Match:   {best_candidate['Experience Match']}%")
print(f"Education Match:    {best_candidate['Education Match']}%")
print(f"Skills Match:       {best_candidate['Skills Match']}%")
print(f"Job Relevance:      {best_candidate['Job Relevance']}%")
print(f"Screening Score:    {best_candidate['Screening Score']}%")
print(f"Recommendation:     {best_candidate['Recommendation']}")
print(f"Matched Skills:     {best_candidate['Matched Skills']}")
print(f"Missing Skills:     {best_candidate['Missing Skills']}")
print(f"Relevant Terms:     {best_candidate['Relevant Terms']}")

BEST ELIGIBLE CANDIDATE
Eligible Rank:      1
ID:                 20674668
Category:           INFORMATION-TECHNOLOGY
Experience Match:   100.0%
Education Match:    100.0%
Skills Match:       35.0%
Job Relevance:      0.0%
Screening Score:    54.0%
Recommendation:     Moderate Match
Matched Skills:     python, sql, statistics
Missing Skills:     pandas, numpy, data analysis, data visualization, machine learning, predictive modeling, scikit-learn
Relevant Terms:     


In [304]:
# ============================================================
# IMPROVED JOB RELEVANCE
# ============================================================

related_terms = {
    "data science": {
        "data science",
        "data scientist",
        "machine learning",
        "predictive modeling",
        "predictive analytics",
        "statistical modeling"
    },

    "statistical modeling": {
        "statistical modeling",
        "statistical analysis",
        "statistics",
        "regression",
        "predictive modeling"
    },

    "data analyst": {
        "data analyst",
        "data analysis",
        "data analytics",
        "analytics",
        "business analytics"
    },

    "data analysis": {
        "data analysis",
        "data analyst",
        "data analytics",
        "analytics",
        "statistical analysis"
    }
}


def calculate_job_relevance(resume_text, jd_terms):

    resume_text = str(resume_text).lower()

    matched_terms = []

    for jd_term in jd_terms:

        jd_term_lower = jd_term.lower()

        # Exact match
        if jd_term_lower in resume_text:
            matched_terms.append(jd_term)
            continue

        # Related concept match
        related = related_terms.get(jd_term_lower, set())

        if any(term in resume_text for term in related):
            matched_terms.append(jd_term)

    if len(jd_terms) == 0:
        return 0.0

    score = (len(set(matched_terms)) / len(jd_terms)) * 100

    return round(score, 2)

In [305]:
candidate = df[df["ID"] == 20674668].iloc[0]

new_relevance = calculate_job_relevance(
    candidate["Cleaned_Resume"],
    relevant_terms
)

print("Candidate ID:", candidate["ID"])
print("New Job Relevance:", new_relevance)

Candidate ID: 20674668
New Job Relevance: 75.0


In [306]:
candidate = df[df["ID"] == 21156767].iloc[0]

new_relevance = calculate_job_relevance(
    candidate["Cleaned_Resume"],
    relevant_terms
)

print("Candidate ID:", candidate["ID"])
print("New Job Relevance:", new_relevance)

Candidate ID: 21156767
New Job Relevance: 100.0


In [307]:
# ============================================================
# RECALCULATE JOB RELEVANCE FOR ALL RESUMES
# ============================================================

df["Job Relevance_New"] = df["Cleaned_Resume"].apply(
    lambda text: calculate_job_relevance(
        text,
        relevant_terms
    )
)

print("===== NEW JOB RELEVANCE DISTRIBUTION =====")
print(df["Job Relevance_New"].value_counts().sort_index())

print("\n===== NEW JOB RELEVANCE STATISTICS =====")
print(df["Job Relevance_New"].describe())

===== NEW JOB RELEVANCE DISTRIBUTION =====
Job Relevance_New
0.0      2063
25.0      111
50.0      246
75.0       55
100.0       6
Name: count, dtype: int64

===== NEW JOB RELEVANCE STATISTICS =====
count    2481.000000
mean        7.980653
std        19.004807
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max       100.000000
Name: Job Relevance_New, dtype: float64


In [308]:
print(
    df.loc[
        df["ID"].isin([20674668, 21156767]),
        ["ID", "Category", "Education_Match_New", "Job Relevance_New"]
    ]
)

            ID                Category  Education_Match_New  Job Relevance_New
291   20674668  INFORMATION-TECHNOLOGY                  0.0               75.0
1217  21156767              CONSULTANT                100.0              100.0


In [309]:
ranking_df = ranking_df.drop(
    columns=["Job Relevance"],
    errors="ignore"
)

ranking_df = ranking_df.merge(
    df[["ID", "Job Relevance_New"]],
    on="ID",
    how="left"
)

ranking_df["Job Relevance"] = ranking_df["Job Relevance_New"]

ranking_df.drop(
    columns=["Job Relevance_New"],
    inplace=True
)

In [310]:
print(
    ranking_df.loc[
        ranking_df["ID"].isin([20674668, 21156767]),
        ["ID", "Job Relevance"]
    ]
)

         ID  Job Relevance
2  20674668           75.0
4  21156767          100.0


In [311]:
ranking_df["Screening Score"] = (
    ranking_df["Experience Match"] * 0.20 +
    ranking_df["Education Match"] * 0.20 +
    ranking_df["Skills Match"] * 0.40 +
    ranking_df["Job Relevance"] * 0.20
).round(2)

ranking_df["Overall Match"] = ranking_df["Screening Score"]

In [312]:
def get_recommendation(score):
    if score >= 70:
        return "Good Match"
    elif score >= 50:
        return "Moderate Match"
    else:
        return "Weak Match"


ranking_df["Recommendation"] = (
    ranking_df["Screening Score"]
    .apply(get_recommendation)
)

In [313]:
eligible_df = (
    ranking_df[
        ranking_df["Eligibility"] == "Eligible"
    ]
    .sort_values("Screening Score", ascending=False)
    .reset_index(drop=True)
)

eligible_df["Eligible Rank"] = eligible_df.index + 1

In [318]:
# ============================================================
# UPDATE EDUCATION MATCH WITH NEW EDUCATION MATCH
# ============================================================

ranking_df = ranking_df.drop(
    columns=["Education Match"],
    errors="ignore"
)

ranking_df = ranking_df.merge(
    df[["ID", "Education_Match_New"]],
    on="ID",
    how="left"
)

ranking_df["Education Match"] = ranking_df["Education_Match_New"]

ranking_df.drop(
    columns=["Education_Match_New"],
    inplace=True
)

In [319]:
display(
    ranking_df[
        ranking_df["ID"].isin([20674668, 21156767])
    ][
        [
            "ID",
            "Category",
            "Education Match",
            "Job Relevance"
        ]
    ]
)

,ID,Category,Education Match,Job Relevance
2,20674668,INFORMATION-TECHNOLOGY,0.0,75.0
4,21156767,CONSULTANT,100.0,100.0


In [320]:
def get_eligibility(row):

    experience_gap = row["Experience Match"] < 100
    education_gap = row["Education Match"] < 100

    if not experience_gap and not education_gap:
        return "Eligible"

    elif experience_gap and education_gap:
        return "Experience + Education Gap"

    elif experience_gap:
        return "Experience Gap"

    else:
        return "Education Gap"


ranking_df["Eligibility"] = ranking_df.apply(
    get_eligibility,
    axis=1
)

In [321]:
display(
    ranking_df[
        ranking_df["ID"].isin([20674668, 21156767])
    ][
        [
            "ID",
            "Experience Match",
            "Education Match",
            "Job Relevance",
            "Eligibility"
        ]
    ]
)

,ID,Experience Match,Education Match,Job Relevance,Eligibility
2,20674668,100.0,0.0,75.0,Education Gap
4,21156767,100.0,100.0,100.0,Eligible


In [322]:
ranking_df["Screening Score"] = (
    ranking_df["Experience Match"] * 0.20 +
    ranking_df["Education Match"] * 0.20 +
    ranking_df["Skills Match"] * 0.40 +
    ranking_df["Job Relevance"] * 0.20
).round(2)

ranking_df["Overall Match"] = ranking_df["Screening Score"]

In [323]:
ranking_df["Recommendation"] = (
    ranking_df["Screening Score"]
    .apply(get_recommendation)
)

In [324]:
eligible_df = (
    ranking_df[
        ranking_df["Eligibility"] == "Eligible"
    ]
    .sort_values(
        "Screening Score",
        ascending=False
    )
    .reset_index(drop=True)
)

eligible_df["Eligible Rank"] = eligible_df.index + 1

In [325]:
display(
    eligible_df.head(15)[
        [
            "Eligible Rank",
            "ID",
            "Category",
            "Experience Match",
            "Education Match",
            "Skills Match",
            "Job Relevance",
            "Screening Score",
            "Recommendation",
            "Matched Skills",
            "Missing Skills",
            "Relevant Terms"
        ]
    ]
)

,Eligible Rank,ID,Category,Experience Match,Education Match,Skills Match,Job Relevance,Screening Score,Recommendation,Matched Skills,Missing Skills,Relevant Terms
0,1,21156767,CONSULTANT,100.0,100.0,51.0,100.0,80.4,Good Match,"python, sql, data analysis, data visualization, machine learning","pandas, numpy, statistics, predictive modeling, scikit-learn","data science, data analysis, machine learning"
1,2,29839396,BANKING,100.0,100.0,35.0,75.0,69.0,Moderate Match,"python, data analysis, statistics","sql, pandas, numpy, data visualization, machine learning, predictive modeling, scikit-learn",data analysis
2,3,32042584,BUSINESS-DEVELOPMENT,100.0,100.0,20.0,100.0,68.0,Moderate Match,"sql, statistics","python, pandas, numpy, data analysis, data visualization, machine learning, predictive modeling,...",data science
3,4,11813872,AGRICULTURE,100.0,100.0,31.0,75.0,67.4,Moderate Match,"python, sql, pandas","numpy, data analysis, statistics, data visualization, machine learning, predictive modeling, sci...",data science
4,5,18067556,INFORMATION-TECHNOLOGY,100.0,100.0,31.0,75.0,67.4,Moderate Match,"python, sql, data analysis","pandas, numpy, statistics, data visualization, machine learning, predictive modeling, scikit-learn",data analysis
5,6,11890896,ENGINEERING,100.0,100.0,28.0,75.0,66.2,Moderate Match,"sql, data analysis, statistics","python, pandas, numpy, data visualization, machine learning, predictive modeling, scikit-learn",data analysis
6,7,20566550,ENGINEERING,100.0,100.0,35.0,50.0,64.0,Moderate Match,"python, sql, statistics","pandas, numpy, data analysis, data visualization, machine learning, predictive modeling, scikit-...",data science
7,8,12374933,CONSULTANT,100.0,100.0,20.0,75.0,63.0,Moderate Match,"sql, statistics","python, pandas, numpy, data analysis, data visualization, machine learning, predictive modeling,...",data analyst
8,9,20393721,ACCOUNTANT,100.0,100.0,20.0,75.0,63.0,Moderate Match,"data analysis, statistics","python, sql, pandas, numpy, data visualization, machine learning, predictive modeling, scikit-learn",data analysis
9,10,12526702,CONSULTANT,100.0,100.0,20.0,75.0,63.0,Moderate Match,"data analysis, statistics","python, sql, pandas, numpy, data visualization, machine learning, predictive modeling, scikit-learn",data analysis


In [326]:
# ============================================================
# FINAL RANKING PIPELINE
# ============================================================

# 1. Recalculate Screening Score
ranking_df["Screening Score"] = (
    ranking_df["Experience Match"] * 0.20 +
    ranking_df["Education Match"] * 0.20 +
    ranking_df["Skills Match"] * 0.40 +
    ranking_df["Job Relevance"] * 0.20
).round(2)


# 2. Recommendation
def get_recommendation(score):
    if score >= 70:
        return "Good Match"
    elif score >= 50:
        return "Moderate Match"
    else:
        return "Weak Match"


ranking_df["Recommendation"] = (
    ranking_df["Screening Score"]
    .apply(get_recommendation)
)


# 3. Recalculate eligibility
def get_eligibility(row):

    experience_gap = row["Experience Match"] < 100
    education_gap = row["Education Match"] < 100

    if not experience_gap and not education_gap:
        return "Eligible"

    elif experience_gap and education_gap:
        return "Experience + Education Gap"

    elif experience_gap:
        return "Experience Gap"

    else:
        return "Education Gap"


ranking_df["Eligibility"] = ranking_df.apply(
    get_eligibility,
    axis=1
)


# 4. Create final eligible ranking
eligible_df = (
    ranking_df[
        ranking_df["Eligibility"] == "Eligible"
    ]
    .sort_values(
        "Screening Score",
        ascending=False
    )
    .reset_index(drop=True)
)

eligible_df["Eligible Rank"] = eligible_df.index + 1


# 5. Display final ranking
display(
    eligible_df.head(15)[
        [
            "Eligible Rank",
            "ID",
            "Category",
            "Experience Match",
            "Education Match",
            "Skills Match",
            "Job Relevance",
            "Screening Score",
            "Recommendation",
            "Matched Skills",
            "Missing Skills",
            "Relevant Terms"
        ]
    ]
)

,Eligible Rank,ID,Category,Experience Match,Education Match,Skills Match,Job Relevance,Screening Score,Recommendation,Matched Skills,Missing Skills,Relevant Terms
0,1,21156767,CONSULTANT,100.0,100.0,51.0,100.0,80.4,Good Match,"python, sql, data analysis, data visualization, machine learning","pandas, numpy, statistics, predictive modeling, scikit-learn","data science, data analysis, machine learning"
1,2,29839396,BANKING,100.0,100.0,35.0,75.0,69.0,Moderate Match,"python, data analysis, statistics","sql, pandas, numpy, data visualization, machine learning, predictive modeling, scikit-learn",data analysis
2,3,32042584,BUSINESS-DEVELOPMENT,100.0,100.0,20.0,100.0,68.0,Moderate Match,"sql, statistics","python, pandas, numpy, data analysis, data visualization, machine learning, predictive modeling,...",data science
3,4,11813872,AGRICULTURE,100.0,100.0,31.0,75.0,67.4,Moderate Match,"python, sql, pandas","numpy, data analysis, statistics, data visualization, machine learning, predictive modeling, sci...",data science
4,5,18067556,INFORMATION-TECHNOLOGY,100.0,100.0,31.0,75.0,67.4,Moderate Match,"python, sql, data analysis","pandas, numpy, statistics, data visualization, machine learning, predictive modeling, scikit-learn",data analysis
5,6,11890896,ENGINEERING,100.0,100.0,28.0,75.0,66.2,Moderate Match,"sql, data analysis, statistics","python, pandas, numpy, data visualization, machine learning, predictive modeling, scikit-learn",data analysis
6,7,20566550,ENGINEERING,100.0,100.0,35.0,50.0,64.0,Moderate Match,"python, sql, statistics","pandas, numpy, data analysis, data visualization, machine learning, predictive modeling, scikit-...",data science
7,8,12374933,CONSULTANT,100.0,100.0,20.0,75.0,63.0,Moderate Match,"sql, statistics","python, pandas, numpy, data analysis, data visualization, machine learning, predictive modeling,...",data analyst
8,9,20393721,ACCOUNTANT,100.0,100.0,20.0,75.0,63.0,Moderate Match,"data analysis, statistics","python, sql, pandas, numpy, data visualization, machine learning, predictive modeling, scikit-learn",data analysis
9,10,12526702,CONSULTANT,100.0,100.0,20.0,75.0,63.0,Moderate Match,"data analysis, statistics","python, sql, pandas, numpy, data visualization, machine learning, predictive modeling, scikit-learn",data analysis


In [327]:
display(
    ranking_df[
        ranking_df["ID"].isin([21156767, 20674668])
    ][
        [
            "ID",
            "Category",
            "Experience Match",
            "Education Match",
            "Skills Match",
            "Job Relevance",
            "Screening Score",
            "Recommendation",
            "Eligibility"
        ]
    ]
)

,ID,Category,Experience Match,Education Match,Skills Match,Job Relevance,Screening Score,Recommendation,Eligibility
2,20674668,INFORMATION-TECHNOLOGY,100.0,0.0,35.0,75.0,49.0,Weak Match,Education Gap
4,21156767,CONSULTANT,100.0,100.0,51.0,100.0,80.4,Good Match,Eligible


In [328]:
def generate_explanation(row):
    
    matched = str(row["Matched Skills"])
    missing = str(row["Missing Skills"])
    relevant = str(row["Relevant Terms"])

    # Convert empty values
    if matched.strip() == "":
        matched_text = "none"
        matched_count = 0
    else:
        matched_list = [x.strip() for x in matched.split(",") if x.strip()]
        matched_count = len(matched_list)
        matched_text = ", ".join(matched_list)

    if missing.strip() == "":
        missing_text = "none"
    else:
        missing_text = missing

    if relevant.strip() == "":
        relevance_text = "No strong job-specific terms detected."
    else:
        relevance_text = f"Relevant job terms: {relevant}"

    explanation = (
        f"Candidate has an experience match of {row['Experience Match']:.1f}% "
        f"and an education match of {row['Education Match']:.1f}%. "
        f"The candidate matches {matched_count} required skills "
        f"({matched_text}). "
        f"Skills match is {row['Skills Match']:.1f}%. "
        f"Job relevance is {row['Job Relevance']:.1f}%. "
        f"Missing skills include: {missing_text}. "
        f"{relevance_text} "
        f"Overall screening score is {row['Screening Score']:.1f}%, "
        f"with recommendation: {row['Recommendation']}."
    )

    return explanation

In [329]:
ranking_df["Explanation"] = ranking_df.apply(
    generate_explanation,
    axis=1
)

In [330]:
candidate = ranking_df[
    ranking_df["ID"] == 21156767
].iloc[0]

print("=" * 80)
print("EXPLANATION VERIFICATION")
print("=" * 80)

print(f"ID:                 {candidate['ID']}")
print(f"Experience Match:   {candidate['Experience Match']}%")
print(f"Education Match:    {candidate['Education Match']}%")
print(f"Skills Match:       {candidate['Skills Match']}%")
print(f"Job Relevance:      {candidate['Job Relevance']}%")
print(f"Screening Score:    {candidate['Screening Score']}%")
print(f"Matched Skills:     {candidate['Matched Skills']}")
print(f"Missing Skills:     {candidate['Missing Skills']}")
print(f"Relevant Terms:     {candidate['Relevant Terms']}")
print()
print("EXPLANATION:")
print(candidate["Explanation"])

EXPLANATION VERIFICATION
ID:                 21156767
Experience Match:   100.0%
Education Match:    100.0%
Skills Match:       51.0%
Job Relevance:      100.0%
Screening Score:    80.4%
Matched Skills:     python, sql, data analysis, data visualization, machine learning
Missing Skills:     pandas, numpy, statistics, predictive modeling, scikit-learn
Relevant Terms:     data science, data analysis, machine learning

EXPLANATION:
Candidate has an experience match of 100.0% and an education match of 100.0%. The candidate matches 5 required skills (python, sql, data analysis, data visualization, machine learning). Skills match is 51.0%. Job relevance is 100.0%. Missing skills include: pandas, numpy, statistics, predictive modeling, scikit-learn. Relevant job terms: data science, data analysis, machine learning Overall screening score is 80.4%, with recommendation: Good Match.


In [331]:
for candidate_id in [21156767, 20674668]:

    candidate = ranking_df[
        ranking_df["ID"] == candidate_id
    ].iloc[0]

    print("=" * 80)
    print(f"ID: {candidate_id}")
    print("=" * 80)

    print(f"Experience Match: {candidate['Experience Match']}%")
    print(f"Education Match:  {candidate['Education Match']}%")
    print(f"Skills Match:     {candidate['Skills Match']}%")
    print(f"Job Relevance:    {candidate['Job Relevance']}%")
    print(f"Screening Score:  {candidate['Screening Score']}%")
    print(f"Eligibility:      {candidate['Eligibility']}")
    print()
    print("Explanation:")
    print(candidate["Explanation"])

ID: 21156767
Experience Match: 100.0%
Education Match:  100.0%
Skills Match:     51.0%
Job Relevance:    100.0%
Screening Score:  80.4%
Eligibility:      Eligible

Explanation:
Candidate has an experience match of 100.0% and an education match of 100.0%. The candidate matches 5 required skills (python, sql, data analysis, data visualization, machine learning). Skills match is 51.0%. Job relevance is 100.0%. Missing skills include: pandas, numpy, statistics, predictive modeling, scikit-learn. Relevant job terms: data science, data analysis, machine learning Overall screening score is 80.4%, with recommendation: Good Match.
ID: 20674668
Experience Match: 100.0%
Education Match:  0.0%
Skills Match:     35.0%
Job Relevance:    75.0%
Screening Score:  49.0%
Eligibility:      Education Gap

Explanation:
Candidate has an experience match of 100.0% and an education match of 0.0%. The candidate matches 3 required skills (python, sql, statistics). Skills match is 35.0%. Job relevance is 75.0%. M

--------UI-----------

In [338]:
import streamlit as st

st.set_page_config(
    page_title="Resume Screening Tool",
    page_icon="📄",
    layout="wide"
)

st.title("📄 Resume Screening Tool")

st.write(
    "Screen and rank candidates based on experience, education, "
    "skills, and job relevance."
)

st.subheader("Job Description")

job_description = st.text_area(
    "Enter the Job Description",
    height=250,
    placeholder="Paste the job description here..."
)

if st.button("🔍 Screen Candidates"):

    if not job_description.strip():
        st.warning("Please enter a job description.")

    else:
        st.success("Job description received!")

        st.write("Screening candidates...")

2026-08-13 03:33:51.098 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-13 03:33:51.101 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-13 03:33:51.102 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-13 03:33:51.103 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-13 03:33:51.105 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-13 03:33:51.106 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-13 03:33:51.107 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-13 03:33:51.109 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [341]:
print(type(df))
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nRanking columns:")
print(ranking_df.columns.tolist())

print("\nFunctions:")
print([
    name for name in [
        "extract_education",
        "calculate_job_relevance",
        "get_eligibility",
        "get_recommendation",
        "generate_explanation"
    ]
    if name in globals()
])

<class 'pandas.core.frame.DataFrame'>
(2481, 7)

Columns:
['ID', 'Resume_str', 'Resume_html', 'Category', 'Cleaned_Resume', 'Education_Match_New', 'Job Relevance_New']

Ranking columns:
['Rank', 'ID', 'Category', 'Experience Match', 'Skills Match', 'Overall Match', 'Recommendation', 'Matched Skills', 'Missing Skills', 'Relevant Terms', 'Explanation', 'Eligibility', 'Screening Score', 'Job Relevance', 'Education Match']

Functions:
['extract_education', 'calculate_job_relevance', 'get_eligibility', 'get_recommendation', 'generate_explanation']


In [342]:
# ============================================================
# FINAL PIPELINE CHECK
# ============================================================

print("DF shape:", df.shape)
print("DF columns:")
print(df.columns.tolist())

print("\nRanking columns:")
print(ranking_df.columns.tolist())

print("\nEligible candidates:", len(eligible_df))

print("\n===== TOP 10 ELIGIBLE CANDIDATES =====")

display(
    eligible_df.head(10)[
        [
            "Eligible Rank",
            "ID",
            "Category",
            "Experience Match",
            "Education Match",
            "Skills Match",
            "Job Relevance",
            "Screening Score",
            "Recommendation",
            "Matched Skills",
            "Missing Skills",
            "Relevant Terms",
            "Eligibility"
        ]
    ]
)

DF shape: (2481, 7)
DF columns:
['ID', 'Resume_str', 'Resume_html', 'Category', 'Cleaned_Resume', 'Education_Match_New', 'Job Relevance_New']

Ranking columns:
['Rank', 'ID', 'Category', 'Experience Match', 'Skills Match', 'Overall Match', 'Recommendation', 'Matched Skills', 'Missing Skills', 'Relevant Terms', 'Explanation', 'Eligibility', 'Screening Score', 'Job Relevance', 'Education Match']

Eligible candidates: 155

===== TOP 10 ELIGIBLE CANDIDATES =====


,Eligible Rank,ID,Category,Experience Match,Education Match,Skills Match,Job Relevance,Screening Score,Recommendation,Matched Skills,Missing Skills,Relevant Terms,Eligibility
0,1,21156767,CONSULTANT,100.0,100.0,51.0,100.0,80.4,Good Match,"python, sql, data analysis, data visualization, machine learning","pandas, numpy, statistics, predictive modeling, scikit-learn","data science, data analysis, machine learning",Eligible
1,2,29839396,BANKING,100.0,100.0,35.0,75.0,69.0,Moderate Match,"python, data analysis, statistics","sql, pandas, numpy, data visualization, machine learning, predictive modeling, scikit-learn",data analysis,Eligible
2,3,32042584,BUSINESS-DEVELOPMENT,100.0,100.0,20.0,100.0,68.0,Moderate Match,"sql, statistics","python, pandas, numpy, data analysis, data visualization, machine learning, predictive modeling,...",data science,Eligible
3,4,11813872,AGRICULTURE,100.0,100.0,31.0,75.0,67.4,Moderate Match,"python, sql, pandas","numpy, data analysis, statistics, data visualization, machine learning, predictive modeling, sci...",data science,Eligible
4,5,18067556,INFORMATION-TECHNOLOGY,100.0,100.0,31.0,75.0,67.4,Moderate Match,"python, sql, data analysis","pandas, numpy, statistics, data visualization, machine learning, predictive modeling, scikit-learn",data analysis,Eligible
5,6,11890896,ENGINEERING,100.0,100.0,28.0,75.0,66.2,Moderate Match,"sql, data analysis, statistics","python, pandas, numpy, data visualization, machine learning, predictive modeling, scikit-learn",data analysis,Eligible
6,7,20566550,ENGINEERING,100.0,100.0,35.0,50.0,64.0,Moderate Match,"python, sql, statistics","pandas, numpy, data analysis, data visualization, machine learning, predictive modeling, scikit-...",data science,Eligible
7,8,12374933,CONSULTANT,100.0,100.0,20.0,75.0,63.0,Moderate Match,"sql, statistics","python, pandas, numpy, data analysis, data visualization, machine learning, predictive modeling,...",data analyst,Eligible
8,9,20393721,ACCOUNTANT,100.0,100.0,20.0,75.0,63.0,Moderate Match,"data analysis, statistics","python, sql, pandas, numpy, data visualization, machine learning, predictive modeling, scikit-learn",data analysis,Eligible
9,10,12526702,CONSULTANT,100.0,100.0,20.0,75.0,63.0,Moderate Match,"data analysis, statistics","python, sql, pandas, numpy, data visualization, machine learning, predictive modeling, scikit-learn",data analysis,Eligible


In [343]:
ranking_df["Screening Score"] = (
    ranking_df["Experience Match"] * 0.20 +
    ranking_df["Education Match"] * 0.20 +
    ranking_df["Skills Match"] * 0.40 +
    ranking_df["Job Relevance"] * 0.20
).round(2)

In [344]:
print(generate_explanation(
    eligible_df.iloc[0]
))

Candidate has an experience match of 100.0% and an education match of 100.0%. The candidate matches 5 required skills (python, sql, data analysis, data visualization, machine learning). Skills match is 51.0%. Job relevance is 100.0%. Missing skills include: pandas, numpy, statistics, predictive modeling, scikit-learn. Relevant job terms: data science, data analysis, machine learning Overall screening score is 80.4%, with recommendation: Good Match.


In [345]:
%who

DATA_PATH	 EDUCATION_ALIASES	 EDUCATION_DIRECT	 EDUCATION_RELATED	 best_candidate	 calculate_education_match	 calculate_final_skill_score	 calculate_job_relevance	 candidate	 
candidate_18448085	 candidate_id	 check_ids	 col	 datetime	 detailed_skill_match	 df	 edu	 education	 
education_check	 education_match	 education_pattern	 education_scores	 education_start	 education_test	 education_text	 eligible_df	 expected	 
explain_candidate	 explanation	 explanations	 extract_education	 extract_experience	 extract_job_relevance	 functions_to_check	 generate_explanation	 get_eligibility	 
get_recommendation	 i	 inspect	 job_description	 job_profile	 keyword	 keywords	 match	 matched_terms	 
name	 new_relevance	 normalize_education_fields	 np	 obj	 os	 pd	 pos	 ranking_df	 
re	 related_terms	 relevance_score	 relevant_terms	 required_education	 required_experience	 result	 results	 resume	 
resume_text	 row	 score	 score_cols	 screen_candidate	 screening_results	 show_candidate	 skill_detail

In [346]:
import inspect

print(inspect.getsource(extract_education))
print(inspect.getsource(calculate_education_match))
print(inspect.getsource(extract_experience))
print(inspect.getsource(calculate_job_relevance))
print(inspect.getsource(calculate_final_skill_score))
print(inspect.getsource(get_eligibility))
print(inspect.getsource(get_recommendation))
print(inspect.getsource(generate_explanation))

def extract_education(resume_text):
    text = str(resume_text).lower()

    # ---------------------------------------------------------
    # 1. Try to isolate the Education section
    # ---------------------------------------------------------
    education_section = text

    education_patterns = [
        r'\beducation\b(.*?)(?=\b(?:skills|experience|work experience|'
        r'professional experience|employment|projects|certifications|'
        r'achievements|summary|objective)\b)',
        r'\bqualification[s]?\b(.*?)(?=\b(?:skills|experience|work experience|'
        r'employment|projects|certifications)\b)'
    ]

    for pattern in education_patterns:
        match = re.search(pattern, text, re.DOTALL)

        if match:
            education_section = match.group(1)
            break

    # ---------------------------------------------------------
    # 2. Extract degrees
    # ---------------------------------------------------------
    degrees = []

    degree_patterns = 